In [ ]:

# INSTALACJA WYMAGANYCH BIBLIOTEK

import sys
import subprocess

REQUIRED_PACKAGES = [
    "pandas==2.2.2",
    "numpy==1.26.4",
    "openpyxl==3.1.5",
    "scikit-learn==1.5.1",
    "xgboost==3.4.1",
    "scipy==1.13.1",
    "requests==2.32.3",
    "beautifulsoup4==4.12.3",
    "pymupdf==1.28.2",
    "matplotlib"
]

print("=" * 70)
print("INSTALACJA / WERYFIKACJA WYMAGANYCH BIBLIOTEK")
print("=" * 70)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    *REQUIRED_PACKAGES
])

print("\\nBiblioteki są zainstalowane. Rozpoczynam wykonywanie badania.\\n")

# W skrypcie standalone funkcja display pochodzi z IPython.
try:
    from IPython.display import display
except ImportError:
    def display(x):
        print(x)


In [ ]:
import re
import time
import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

START_YEAR = 2015
END_YEAR = 2024
OUTPUT_FILE = "dane_finansowe_gpw_2015_2024.xlsx"

COMPANIES = [
    {"company": "ORLEN", "ticker": "PKN", "slug": "ORLEN"},
    {"company": "KGHM Polska Miedź", "ticker": "KGH", "slug": "KGHM"},
    {"company": "Orange Polska", "ticker": "OPL", "slug": "ORANGE"},
    {"company": "Asseco Poland", "ticker": "ACP", "slug": "ASSECO-POLAND"},
    {"company": "Budimex", "ticker": "BDX", "slug": "BUDIMEX"},
    {"company": "LPP", "ticker": "LPP", "slug": "LPP"},
    {"company": "CD Projekt", "ticker": "CDR", "slug": "CD-PROJEKT"},
    {"company": "Cyfrowy Polsat", "ticker": "CPS", "slug": "CYFROWY-POLSAT"},
    {"company": "PGE", "ticker": "PGE", "slug": "PGE"},
    {"company": "Grupa Kęty", "ticker": "KTY", "slug": "KETY"},
]

BASE_URL = "https://www.biznesradar.pl/raporty-finansowe-rachunek-zyskow-i-strat/{slug},Q"

HEADERS = {
    "User-Agent": "Mozilla/5.0"
}

def clean_text(text):
    return re.sub(r"\s+", " ", text.replace("\xa0", " ")).strip()

def parse_number(text):
    text = clean_text(text)
    if not text or text in ["-", "—", "b.d.", "n/a"]:
        return None

    text = re.split(r"\br/r\b|\bk/k\b", text, maxsplit=1)[0].strip()
    match = re.search(r"[-+]?\d[\d\s]*(?:[.,]\d+)?", text)

    if not match:
        return None

    value = match.group(0).replace(" ", "").replace(",", ".")

    try:
        return float(value)
    except:
        return None

def parse_period(text):
    match = re.search(r"(20\d{2})/Q([1-4])", text)
    if match:
        return int(match.group(1)), f"Q{match.group(2)}"
    return None

def find_financial_table(soup):
    for table in soup.find_all("table"):
        text = clean_text(table.get_text(" ", strip=True))
        if "Data publikacji" in text and "Zysk netto" in text:
            return table
    return None

def get_rows(table):
    rows = {}

    for tr in table.find_all("tr"):
        cells = tr.find_all(["th", "td"])
        texts = [clean_text(c.get_text(" ", strip=True)) for c in cells]

        if len(texts) > 1:
            rows[texts[0]] = texts[1:]

    return rows

def find_row(rows, phrases):
    for key, values in rows.items():
        for phrase in phrases:
            if phrase.lower() in key.lower():
                return values
    return None

all_records = []
errors = []

for company in COMPANIES:

    url = BASE_URL.format(slug=company["slug"])

    print("Pobieram:", company["company"])

    try:
        response = requests.get(url, headers=HEADERS, timeout=30)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, "html.parser")

        table = find_financial_table(soup)

        if table is None:
            raise Exception("Nie znaleziono tabeli danych")

        rows = get_rows(table)

        periods = []

        for tr in table.find_all("tr"):
            texts = [
                clean_text(c.get_text(" ", strip=True))
                for c in tr.find_all(["th", "td"])
            ]

            found = [x for x in texts if re.search(r"20\d{2}/Q[1-4]", x)]

            if found:
                periods = found
                break

        publication = find_row(rows, ["Data publikacji"])
        revenue = find_row(rows, ["Przychody ze sprzedaży"])
        ebit = find_row(rows, ["EBIT", "Zysk operacyjny"])
        net_income = find_row(rows, ["Zysk netto"])

        if revenue is None:
            raise Exception("Brak przychodów")

        if ebit is None:
            raise Exception("Brak EBIT")

        if net_income is None:
            raise Exception("Brak zysku netto")

        n = min(len(periods), len(revenue), len(ebit), len(net_income))

        for i in range(n):

            parsed = parse_period(periods[i])

            if parsed is None:
                continue

            year, quarter = parsed

            if not START_YEAR <= year <= END_YEAR:
                continue

            pub_date = publication[i] if publication and i < len(publication) else None

            all_records.append([
                company["company"],
                company["ticker"],
                year,
                quarter,
                pub_date,
                parse_number(revenue[i]),
                parse_number(ebit[i]),
                parse_number(net_income[i]),
                url
            ])

        print("OK")

    except Exception as e:
        print("BŁĄD:", e)
        errors.append([company["company"], str(e)])

    time.sleep(1.5)

print("Liczba rekordów:", len(all_records))

In [ ]:
from openpyxl import Workbook
from openpyxl.styles import Font
from openpyxl.utils import get_column_letter

OUTPUT_FILE = "dane_finansowe_gpw_2015_2024.xlsx"

wb = Workbook()
ws = wb.active
ws.title = "Dane kwartalne"

headers = [
    "Spółka",
    "Ticker",
    "Rok",
    "Kwartał",
    "Data publikacji",
    "Przychody ze sprzedaży",
    "EBIT",
    "Zysk netto",
    "Źródło"
]

ws.append(headers)

for cell in ws[1]:
    cell.font = Font(bold=True)

for record in all_records:
    ws.append(record)

for column_cells in ws.columns:
    max_length = 0
    column_letter = get_column_letter(column_cells[0].column)

    for cell in column_cells:
        try:
            max_length = max(max_length, len(str(cell.value)))
        except:
            pass

    ws.column_dimensions[column_letter].width = min(max_length + 2, 50)

# Arkusz z ewentualnymi błędami
log = wb.create_sheet("Log")
log.append(["Spółka", "Błąd"])

for row in errors:
    log.append(row)

wb.save(OUTPUT_FILE)

print("Zapisano plik:", OUTPUT_FILE)

In [ ]:
# PRZYGOTOWANIE DATASETU DO MODELI ML
import os
from pathlib import Path
import pandas as pd
import numpy as np

# 1. USTAWIENIE FOLDERU ROBOCZEGO
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

FOLDER = BASE_DIR

os.chdir(FOLDER)

print("Aktualny folder roboczy:")
print(os.getcwd())

# 2. NAZWY PLIKÓW

INPUT_FILE = "dane_finansowe_gpw_2015_2024.xlsx"

OUTPUT_FILE = "dane_finansowe_gpw_2015_2024_dataset_ML.xlsx"

# 3. SPRAWDZENIE, CZY PLIK ISTNIEJE

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Nie znaleziono pliku: {INPUT_FILE}\n"
        f"Sprawdź, czy znajduje się w folderze:\n{FOLDER}"
    )

print("\nZnaleziono plik wejściowy:", INPUT_FILE)


# 4. WCZYTANIE DANYCH

df_original = pd.read_excel(
    INPUT_FILE,
    sheet_name="Dane kwartalne"
)

df = df_original.copy()

print("\nLiczba wczytanych rekordów:", len(df))

print("\nPierwsze 5 wierszy:")
print(df.head())


# 5. SPRAWDZENIE WYMAGANYCH KOLUMN

required_columns = [
    "Spółka",
    "Ticker",
    "Rok",
    "Kwartał",
    "Data publikacji",
    "Przychody ze sprzedaży",
    "EBIT",
    "Zysk netto"
]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Brakuje wymaganych kolumn: {missing_columns}"
    )

print("\nWszystkie wymagane kolumny są obecne.")


# 6. KONWERSJA TYPÓW DANYCH

df["Rok"] = pd.to_numeric(
    df["Rok"],
    errors="coerce"
)

df["Przychody ze sprzedaży"] = pd.to_numeric(
    df["Przychody ze sprzedaży"],
    errors="coerce"
)

df["EBIT"] = pd.to_numeric(
    df["EBIT"],
    errors="coerce"
)

df["Zysk netto"] = pd.to_numeric(
    df["Zysk netto"],
    errors="coerce"
)

df["Data publikacji"] = pd.to_datetime(
    df["Data publikacji"],
    errors="coerce"
)



# 7. NUMER KWARTAŁU

df["Numer kwartału"] = (
    df["Kwartał"]
    .astype(str)
    .str.extract(r"(\d)")
    .astype(float)
)

# 8. SORTOWANIE CHRONOLOGICZNE

df = df.sort_values(
    by=[
        "Spółka",
        "Rok",
        "Numer kwartału"
    ]
).reset_index(drop=True)


# 9. SPRAWDZENIE DUPLIKATÓW

duplicates = df.duplicated(
    subset=[
        "Spółka",
        "Rok",
        "Kwartał"
    ]
).sum()

print("\nLiczba duplikatów:", duplicates)


# 10. SPRAWDZENIE BRAKÓW DANYCH

print("\nBraki danych w podstawowych kolumnach:")

print(
    df[
        [
            "Przychody ze sprzedaży",
            "EBIT",
            "Zysk netto"
        ]
    ].isna().sum()
)



In [ ]:
# EKSPLORACYJNA ANALIZA DANYCH (EDA)

import pandas as pd
import numpy as np

# 1. ZMIENNE ANALIZOWANE W EDA

EDA_COLUMNS = [
    "Przychody ze sprzedaży",
    "EBIT",
    "Zysk netto"
]

# 2. PODSTAWOWE INFORMACJE O ZBIORZE

print("=" * 70)
print("EKSPLORACYJNA ANALIZA DANYCH (EDA)")
print("=" * 70)

print(f"\nLiczba obserwacji: {len(df)}")
print(f"Liczba spółek: {df['Spółka'].nunique()}")
print(
    f"Zakres lat: "
    f"{int(df['Rok'].min())}-"
    f"{int(df['Rok'].max())}"
)

# 3. STRUKTURA BADANEJ PRÓBY

company_order = (
    df["Spółka"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

sample_rows = []

for company in company_order:

    company_df = df[
        df["Spółka"] == company
    ]

    sample_rows.append({
        "Spółka": company,
        "Okres obserwacji": (
            f"{int(company_df['Rok'].min())}"
            f"–"
            f"{int(company_df['Rok'].max())}"
        ),
        "Liczba obserwacji kwartalnych":
            len(company_df)
    })


sample_structure = pd.DataFrame(
    sample_rows
)


# Dodanie wiersza łącznego

sample_total = pd.DataFrame([
    {
        "Spółka": "Łącznie",
        "Okres obserwacji": (
            f"{int(df['Rok'].min())}"
            f"–"
            f"{int(df['Rok'].max())}"
        ),
        "Liczba obserwacji kwartalnych":
            len(df)
    }
])


sample_structure_with_total = pd.concat(
    [
        sample_structure,
        sample_total
    ],
    ignore_index=True
)


print("\n" + "=" * 70)
print("STRUKTURA BADANEJ PRÓBY")
print("=" * 70)

print(
    sample_structure_with_total
    .to_string(index=False)
)

. STATYSTYKI OPISOWE

eda_stats = pd.DataFrame({

    "N":
        df[EDA_COLUMNS].count(),

    "Średnia":
        df[EDA_COLUMNS].mean(),

    "Mediana":
        df[EDA_COLUMNS].median(),

    "Odchylenie standardowe":
        df[EDA_COLUMNS].std(),

    "Minimum":
        df[EDA_COLUMNS].min(),

    "Maksimum":
        df[EDA_COLUMNS].max()
})


eda_stats.index.name = "Zmienna"


# Wartości finansowe są wyrażone w tys. zł

eda_stats[
    [
        "Średnia",
        "Mediana",
        "Odchylenie standardowe",
        "Minimum",
        "Maksimum"
    ]
] = (
    eda_stats[
        [
            "Średnia",
            "Mediana",
            "Odchylenie standardowe",
            "Minimum",
            "Maksimum"
        ]
    ]
    .round(0)
)


print("\n" + "=" * 70)
print("STATYSTYKI OPISOWE")
print("=" * 70)

print(eda_stats)

# 5. KWARTYLE

quartiles = pd.DataFrame({

    "Q1":
        df[EDA_COLUMNS].quantile(0.25),

    "Q3":
        df[EDA_COLUMNS].quantile(0.75)
}).round(0)


quartiles.index.name = "Zmienna"


print("\n" + "=" * 70)
print("KWARTYLE")
print("=" * 70)

print(quartiles)

# 6. WARTOŚCI UJEMNE

negative_counts = (
    df[EDA_COLUMNS] < 0
).sum()


negative_share = (
    negative_counts
    / df[EDA_COLUMNS].count()
    * 100
).round(2)

# 7. WARTOŚCI RÓWNE ZERO

zero_counts = (
    df[EDA_COLUMNS] == 0
).sum()


negative_summary = pd.DataFrame({

    "Liczba wartości ujemnych":
        negative_counts,

    "Udział wartości ujemnych (%)":
        negative_share,

    "Liczba wartości równych zero":
        zero_counts
})


negative_summary.index.name = "Zmienna"


print("\n" + "=" * 70)
print("WARTOŚCI UJEMNE I ZEROWE")
print("=" * 70)

print(negative_summary)

# 8. BRAKI DANYCH

missing_summary = pd.DataFrame({

    "Liczba braków":
        df[EDA_COLUMNS].isna().sum(),

    "Udział braków (%)":
        (
            df[EDA_COLUMNS].isna().sum()
            / len(df)
            * 100
        ).round(2)
})


missing_summary.index.name = "Zmienna"


print("\n" + "=" * 70)
print("BRAKI DANYCH")
print("=" * 70)

print(missing_summary)

# 9. DUPLIKATY

full_duplicates = df.duplicated().sum()

print("\n" + "=" * 70)
print("DUPLIKATY")
print("=" * 70)

print(
    "Liczba pełnych zduplikowanych rekordów:",
    full_duplicates
)

# 10. STATYSTYKI WEDŁUG SPÓŁEK

company_stats = (
    df
    .groupby("Spółka")[EDA_COLUMNS]
    .agg([
        "count",
        "mean",
        "median",
        "std",
        "min",
        "max"
    ])
    .round(0)
)


print("\n" + "=" * 70)
print("STATYSTYKI WEDŁUG SPÓŁEK")
print("=" * 70)

print(company_stats)

# 11. ZAPIS WYNIKÓW EDA DO EXCELA

EDA_OUTPUT_FILE = "EDA_podsumowanie.xlsx"


with pd.ExcelWriter(
    EDA_OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    sample_structure_with_total.to_excel(
        writer,
        sheet_name="Struktura próby",
        index=False
    )

    eda_stats.reset_index().to_excel(
        writer,
        sheet_name="Statystyki opisowe",
        index=False
    )

    quartiles.reset_index().to_excel(
        writer,
        sheet_name="Kwartyle",
        index=False
    )

    negative_summary.reset_index().to_excel(
        writer,
        sheet_name="Wartości ujemne",
        index=False
    )

    missing_summary.reset_index().to_excel(
        writer,
        sheet_name="Braki danych",
        index=False
    )

    company_stats.to_excel(
        writer,
        sheet_name="Statystyki spółek"
    )


print("\n" + "=" * 70)
print("EDA ZAKOŃCZONE")
print("=" * 70)

print(
    f"Wyniki zapisano do pliku: "
    f"{EDA_OUTPUT_FILE}"
)

In [ ]:
# RYSUNEK EDA
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter


FIGURE_EDA_FILE = "Wykres_EDA_boxploty.png"

# 1. FORMATOWANIE OSI

def format_axis_number(value, position):

    return (
        f"{value:,.0f}"
        .replace(",", " ")
    )


axis_formatter = FuncFormatter(
    format_axis_number
)

# 2. KOLEJNOŚĆ SPÓŁEK

company_order = (
    df["Spółka"]
    .dropna()
    .drop_duplicates()
    .tolist()
)

# 3. ZMIENNE DO PRZEDSTAWIENIA

plot_variables = [

    (
        "Przychody ze sprzedaży",
        "a) Przychody ze sprzedaży"
    ),

    (
        "EBIT",
        "b) EBIT"
    ),

    (
        "Zysk netto",
        "c) Zysk netto"
    )
]

. UTWORZENIE RYSUNKU

fig, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(14, 18)
)


for ax, (
    variable,
    panel_title
) in zip(
    axes,
    plot_variables
):

    values = [

        df.loc[
            df["Spółka"] == company,
            variable
        ]
        .dropna()
        .to_numpy()

        for company in company_order
    ]


    ax.boxplot(
        values,
        showfliers=True
    )


    ax.set_xticks(
        range(
            1,
            len(company_order) + 1
        )
    )


    ax.set_xticklabels(
        company_order,
        rotation=45,
        ha="right"
    )


    ax.set_title(
        panel_title,
        fontsize=12
    )


    ax.set_xlabel(
        "Spółka"
    )


    ax.set_ylabel(
        "tys. zł"
    )


    ax.yaxis.set_major_formatter(
        axis_formatter
    )


    ax.grid(
        axis="y",
        alpha=0.25
    )

# 5. TYTUŁ CAŁEGO RYSUNKU

fig.suptitle(
    (
        "Rozkład przychodów ze sprzedaży, "
        "EBIT i zysku netto według spółek"
    ),
    fontsize=14
)


fig.tight_layout(
    rect=[
        0,
        0,
        1,
        0.97
    ]
)

# 6. ZAPIS DO PNG

plt.savefig(
    FIGURE_EDA_FILE,
    dpi=300,
    bbox_inches="tight"
)


plt.show()


print(
    f"Zapisano rysunek do pliku: "
    f"{FIGURE_EDA_FILE}"
)

In [ ]:
# 1. GRUPOWANIE PO SPÓŁKACH

group = df.groupby(
    "Spółka",
    sort=False
)

# 12. LAGI DLA PRZYCHODÓW

df["Przychody_lag1"] = group[
    "Przychody ze sprzedaży"
].shift(1)

df["Przychody_lag2"] = group[
    "Przychody ze sprzedaży"
].shift(2)

df["Przychody_lag4"] = group[
    "Przychody ze sprzedaży"
].shift(4)

# 13. LAGI DLA EBIT

df["EBIT_lag1"] = group[
    "EBIT"
].shift(1)

df["EBIT_lag4"] = group[
    "EBIT"
].shift(4)

# 14. LAGI DLA ZYSKU NETTO

df["Zysk_netto_lag1"] = group[
    "Zysk netto"
].shift(1)

df["Zysk_netto_lag4"] = group[
    "Zysk netto"
].shift(4)

# 15. DYNAMIKA PRZYCHODÓW R/R
# Obliczamy ją wyłącznie na podstawie danych,
# które były dostępne przed prognozowanym okresem.
# Przychody_lag1 = poprzedni kwartał
# Przychody_lag5 = odpowiadający mu kwartał rok wcześniej

df["Przychody_lag5"] = group[
    "Przychody ze sprzedaży"
].shift(5)

df["Dynamika_przychodów_rr"] = np.where(

    df["Przychody_lag5"].notna()
    & (df["Przychody_lag5"] != 0),

    (
        df["Przychody_lag1"]
        - df["Przychody_lag5"]
    )
    / np.abs(df["Przychody_lag5"]),

    np.nan
)

# 16. MARŻA EBIT Z POPRZEDNIEGO KWARTAŁU

df["Marża_EBIT_lag1"] = np.where(

    df["Przychody_lag1"] != 0,

    df["EBIT_lag1"]
    / df["Przychody_lag1"],

    np.nan
)

# 17. MARŻA NETTO Z POPRZEDNIEGO KWARTAŁU

df["Marża_netto_lag1"] = np.where(

    df["Przychody_lag1"] != 0,

    df["Zysk_netto_lag1"]
    / df["Przychody_lag1"],

    np.nan
)

# 18. PODZIAŁ NA TRAIN I TEST
# TRAIN: 2015-2022
# TEST: 2023-2024

df["Zbiór"] = np.where(
    df["Rok"] <= 2022,
    "TRAIN",
    "TEST"
)

# 19. SPRAWDZENIE, CZY REKORD JEST GOTOWY DO ML

features_required = [

    "Przychody_lag1",
    "Przychody_lag2",
    "Przychody_lag4",

    "EBIT_lag1",
    "EBIT_lag4",

    "Zysk_netto_lag1",
    "Zysk_netto_lag4",

    "Dynamika_przychodów_rr",

    "Marża_EBIT_lag1",
    "Marża_netto_lag1"
]


df["Gotowe_do_ML"] = np.where(

    df[features_required]
    .notna()
    .all(axis=1),

    "TAK",

    "NIE"
)

# 20. USUNIĘCIE TECHNICZNEGO LAG5

df.drop(
    columns=["Przychody_lag5"],
    inplace=True
)


# 21. USTALENIE KOLEJNOŚCI KOLUMN

columns_order = [

    "Spółka",
    "Ticker",
    "Rok",
    "Kwartał",
    "Numer kwartału",
    "Data publikacji",

    "Przychody ze sprzedaży",
    "EBIT",
    "Zysk netto",

    "Przychody_lag1",
    "Przychody_lag2",
    "Przychody_lag4",

    "EBIT_lag1",
    "EBIT_lag4",

    "Zysk_netto_lag1",
    "Zysk_netto_lag4",

    "Dynamika_przychodów_rr",

    "Marża_EBIT_lag1",
    "Marża_netto_lag1",

    "Zbiór",
    "Gotowe_do_ML"
]


if "Źródło" in df.columns:
    columns_order.append("Źródło")


df_ml = df[
    columns_order
].copy()

# 22. PODSUMOWANIE DATASETU

print("\n==========================================")
print("PODSUMOWANIE DATASETU")
print("==========================================")

print(
    "Wszystkich obserwacji:",
    len(df_ml)
)

print(
    "Gotowych do ML:",
    (df_ml["Gotowe_do_ML"] == "TAK").sum()
)

train_count = (
    (
        df_ml["Zbiór"] == "TRAIN"
    )
    &
    (
        df_ml["Gotowe_do_ML"] == "TAK"
    )
).sum()

test_count = (
    (
        df_ml["Zbiór"] == "TEST"
    )
    &
    (
        df_ml["Gotowe_do_ML"] == "TAK"
    )
).sum()

print(
    "TRAIN gotowych do ML:",
    train_count
)

print(
    "TEST gotowych do ML:",
    test_count
)


print("\nLiczba rekordów według spółek:")

print(
    df_ml.groupby(
        "Spółka"
    ).size()
)

# 23. UTWORZENIE ARKUSZA METODYKA ML

metodyka = pd.DataFrame({

    "Zmienna": [

        "Przychody_lag1",

        "Przychody_lag2",

        "Przychody_lag4",

        "EBIT_lag1",

        "EBIT_lag4",

        "Zysk_netto_lag1",

        "Zysk_netto_lag4",

        "Dynamika_przychodów_rr",

        "Marża_EBIT_lag1",

        "Marża_netto_lag1",

        "Numer kwartału",

        "TRAIN",

        "TEST",

        "Gotowe_do_ML"
    ],

    "Opis": [

        "Przychody ze sprzedaży z poprzedniego kwartału.",

        "Przychody ze sprzedaży sprzed dwóch kwartałów.",

        "Przychody ze sprzedaży z tego samego kwartału poprzedniego roku.",

        "EBIT z poprzedniego kwartału.",

        "EBIT z tego samego kwartału poprzedniego roku.",

        "Zysk netto z poprzedniego kwartału.",

        "Zysk netto z tego samego kwartału poprzedniego roku.",

        "Historyczna dynamika przychodów rok do roku "
        "wyliczona wyłącznie na podstawie danych dostępnych "
        "przed prognozowanym okresem.",

        "Marża EBIT z poprzedniego kwartału, "
        "obliczona jako EBIT(t-1) / Przychody(t-1).",

        "Marża netto z poprzedniego kwartału, "
        "obliczona jako Zysk netto(t-1) / Przychody(t-1).",

        "Numer kwartału od 1 do 4.",

        "Dane z lat 2015-2022 przeznaczone do uczenia modeli.",

        "Dane z lat 2023-2024 przeznaczone do testowania modeli.",

        "TAK oznacza, że obserwacja posiada komplet "
        "wymaganych cech historycznych."
    ]
})

# 24. ZAPIS DO NOWEGO PLIKU EXCEL

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    # Oryginalne dane
    df_original.to_excel(
        writer,
        sheet_name="Dane kwartalne",
        index=False
    )

    # Dataset do ML
    df_ml.to_excel(
        writer,
        sheet_name="Dataset ML",
        index=False
    )

    # Metodyka
    metodyka.to_excel(
        writer,
        sheet_name="Metodyka ML",
        index=False
    )

# 25. INFORMACJA KOŃCOWA

print("\n==========================================")
print("GOTOWE")
print("==========================================")

print("\nUtworzono plik:")
print(OUTPUT_FILE)

print("\nPełna ścieżka:")
print(os.path.abspath(OUTPUT_FILE))

In [ ]:
# MODELOWANIE WYNIKÓW FINANSOWYCH
# Seasonal Naive + Linear Regression + XGBoost

import os
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error

from xgboost import XGBRegressor

# 1. USTAWIENIA
FOLDER = BASE_DIR

os.chdir(FOLDER)

INPUT_FILE = "dane_finansowe_gpw_2015_2024_dataset_ML.xlsx"
OUTPUT_FILE = "wyniki_modeli_ML.xlsx"

print("Folder roboczy:")
print(os.getcwd())

# 2. WCZYTANIE DANYCH

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Dataset ML"
)

print("\nLiczba wszystkich rekordów:", len(df))

# 3. WYBÓR REKORDÓW GOTOWYCH DO ML

data = df[
    df["Gotowe_do_ML"] == "TAK"
].copy()

print("Liczba rekordów gotowych do ML:", len(data))

. PODZIAŁ TRAIN / TEST

train = data[
    data["Zbiór"] == "TRAIN"
].copy()

test = data[
    data["Zbiór"] == "TEST"
].copy()

print("\nTRAIN:", len(train))
print("TEST:", len(test))

# 5. CECHY WEJŚCIOWE

numeric_features = [
    "Numer kwartału",

    "Przychody_lag1",
    "Przychody_lag2",
    "Przychody_lag4",

    "EBIT_lag1",
    "EBIT_lag4",

    "Zysk_netto_lag1",
    "Zysk_netto_lag4",

    "Dynamika_przychodów_rr",

    "Marża_EBIT_lag1",
    "Marża_netto_lag1"
]

categorical_features = [
    "Spółka"
]

features = (
    numeric_features
    + categorical_features
)

# 6. ZMIENNE DOCELOWE

targets = {
    "Przychody": "Przychody ze sprzedaży",
    "EBIT": "EBIT",
    "Zysk_netto": "Zysk netto"
}

# 7. FUNKCJA METRYK

def calculate_metrics(y_true, y_pred):

    mae = mean_absolute_error(
        y_true,
        y_pred
    )

    rmse = np.sqrt(
        mean_squared_error(
            y_true,
            y_pred
        )
    )

    return mae, rmse

# 8. PRZYGOTOWANIE LIST NA WYNIKI

results = []

predictions_all = []

# 9. PĘTLA DLA KAŻDEJ ZMIENNEJ DOCELOWEJ

for target_name, target_column in targets.items():

    print("\n")
    print("=" * 60)
    print("PROGNOZOWANA ZMIENNA:", target_name)
    print("=" * 60)

    X_train = train[features]
    X_test = test[features]

    y_train = train[target_column]
    y_test = test[target_column]

    # 9A. SEASONAL NAIVE

    if target_name == "Przychody":
        seasonal_pred = test["Przychody_lag4"]

    elif target_name == "EBIT":
        seasonal_pred = test["EBIT_lag4"]

    elif target_name == "Zysk_netto":
        seasonal_pred = test["Zysk_netto_lag4"]


    seasonal_mae, seasonal_rmse = calculate_metrics(
        y_test,
        seasonal_pred
    )

    print("\nSeasonal Naive")
    print("MAE :", seasonal_mae)
    print("RMSE:", seasonal_rmse)

    results.append({
        "Zmienna": target_name,
        "Model": "Seasonal Naive",
        "MAE": seasonal_mae,
        "RMSE": seasonal_rmse
    })

    # 9B. LINEAR REGRESSION

    linear_preprocessor = ColumnTransformer(
        transformers=[

            (
                "num",
                StandardScaler(),
                numeric_features
            ),

            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            )

        ]
    )

    linear_model = Pipeline(
        steps=[

            (
                "preprocessor",
                linear_preprocessor
            ),

            (
                "model",
                LinearRegression()
            )

        ]
    )

    linear_model.fit(
        X_train,
        y_train
    )

    linear_pred = linear_model.predict(
        X_test
    )

    linear_mae, linear_rmse = calculate_metrics(
        y_test,
        linear_pred
    )

    print("\nLinear Regression")
    print("MAE :", linear_mae)
    print("RMSE:", linear_rmse)

    results.append({
        "Zmienna": target_name,
        "Model": "Linear Regression",
        "MAE": linear_mae,
        "RMSE": linear_rmse
    })

    # 9C. XGBOOST

    xgb_preprocessor = ColumnTransformer(
        transformers=[

            (
                "num",
                "passthrough",
                numeric_features
            ),

            (
                "cat",
                OneHotEncoder(
                    handle_unknown="ignore"
                ),
                categorical_features
            )

        ]
    )

    xgb_model = Pipeline(
        steps=[

            (
                "preprocessor",
                xgb_preprocessor
            ),

            (
                "model",
                XGBRegressor(
                    n_estimators=300,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.8,
                    colsample_bytree=0.8,
                    objective="reg:squarederror",
                    random_state=42,
                    n_jobs=-1
                )
            )

        ]
    )

    xgb_model.fit(
        X_train,
        y_train
    )

    xgb_pred = xgb_model.predict(
        X_test
    )

    xgb_mae, xgb_rmse = calculate_metrics(
        y_test,
        xgb_pred
    )

    print("\nXGBoost")
    print("MAE :", xgb_mae)
    print("RMSE:", xgb_rmse)

    results.append({
        "Zmienna": target_name,
        "Model": "XGBoost",
        "MAE": xgb_mae,
        "RMSE": xgb_rmse
    })

    # 10. ZAPIS PROGNOZ DLA KAŻDEJ OBSERWACJI

    temp = test[
        [
            "Spółka",
            "Ticker",
            "Rok",
            "Kwartał"
        ]
    ].copy()

    temp["Zmienna"] = target_name

    temp["Wartość rzeczywista"] = y_test.values

    temp["Seasonal Naive"] = seasonal_pred.values

    temp["Linear Regression"] = linear_pred

    temp["XGBoost"] = xgb_pred

    temp["Błąd abs. Seasonal Naive"] = np.abs(
        temp["Wartość rzeczywista"]
        - temp["Seasonal Naive"]
    )

    temp["Błąd abs. Linear Regression"] = np.abs(
        temp["Wartość rzeczywista"]
        - temp["Linear Regression"]
    )

    temp["Błąd abs. XGBoost"] = np.abs(
        temp["Wartość rzeczywista"]
        - temp["XGBoost"]
    )

    predictions_all.append(temp)


# 11. TABELA Z WYNIKAMI

results_df = pd.DataFrame(
    results
)

results_df = results_df.sort_values(
    by=[
        "Zmienna",
        "RMSE"
    ]
)

print("\n")
print("=" * 70)
print("PODSUMOWANIE WYNIKÓW")
print("=" * 70)

print(results_df)


# 12. POŁĄCZENIE WSZYSTKICH PROGNOZ

predictions_df = pd.concat(
    predictions_all,
    ignore_index=True
)

# 13. WYNIKI WEDŁUG SPÓŁEK

company_results = []

for variable in predictions_df["Zmienna"].unique():

    temp = predictions_df[
        predictions_df["Zmienna"] == variable
    ]

    for company in temp["Spółka"].unique():

        company_data = temp[
            temp["Spółka"] == company
        ]

        y_true = company_data[
            "Wartość rzeczywista"
        ]

        for model in [
            "Seasonal Naive",
            "Linear Regression",
            "XGBoost"
        ]:

            y_pred = company_data[
                model
            ]

            mae, rmse = calculate_metrics(
                y_true,
                y_pred
            )

            company_results.append({
                "Spółka": company,
                "Zmienna": variable,
                "Model": model,
                "MAE": mae,
                "RMSE": rmse
            })


company_results_df = pd.DataFrame(
    company_results
)

# 14. WYBÓR NAJLEPSZEGO MODELU

best_models = (
    results_df
    .sort_values(
        ["Zmienna", "RMSE"]
    )
    .groupby(
        "Zmienna"
    )
    .first()
    .reset_index()
)

print("\n")
print("=" * 70)
print("NAJLEPSZY MODEL WG RMSE")
print("=" * 70)

print(best_models)

# 15. ZAPIS DO EXCELA

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    results_df.to_excel(
        writer,
        sheet_name="Wyniki ogólne",
        index=False
    )

    predictions_df.to_excel(
        writer,
        sheet_name="Prognozy",
        index=False
    )

    company_results_df.to_excel(
        writer,
        sheet_name="Wyniki spółki",
        index=False
    )

    best_models.to_excel(
        writer,
        sheet_name="Najlepsze modele",
        index=False
    )


print("\n")
print("=" * 70)
print("GOTOWE")
print("=" * 70)

print("\nPlik wynikowy:")
print(OUTPUT_FILE)

print("\nPełna ścieżka:")
print(os.path.abspath(OUTPUT_FILE))



In [ ]:
import os
import numpy as np
import pandas as pd

# 1. USTAWIENIA

INPUT_FILE = "wyniki_modeli_ML.xlsx"
OUTPUT_FILE = "wyniki_modeli_ML_rozszerzone.xlsx"

# 2. WCZYTANIE PROGNOZ

pred = pd.read_excel(
    INPUT_FILE,
    sheet_name="Prognozy"
)

print("Liczba rekordów:", len(pred))
print(pred.head())

# 3. FUNKCJE METRYK

def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))


def rmse(y_true, y_pred):
    return np.sqrt(
        np.mean((y_true - y_pred) ** 2)
    )


def smape(y_true, y_pred):
    """
    Symmetric Mean Absolute Percentage Error
    Wynik w %
    """
    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    )

    mask = denominator != 0

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            2
            * np.abs(y_pred[mask] - y_true[mask])
            / denominator[mask]
        )
        * 100
    )


def wape(y_true, y_pred):
    """
    Weighted Absolute Percentage Error
    Wynik w %
    """
    denominator = np.sum(
        np.abs(y_true)
    )

    if denominator == 0:
        return np.nan

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        / denominator
        * 100
    )


. NAZWY MODELI

models = [
    "Seasonal Naive",
    "Linear Regression",
    "XGBoost"
]

# 5. WYNIKI OGÓLNE Z DODATKOWYMI METRYKAMI

overall_results = []

for variable in pred["Zmienna"].unique():

    temp = pred[
        pred["Zmienna"] == variable
    ].copy()

    y_true = temp[
        "Wartość rzeczywista"
    ].to_numpy(dtype=float)

    for model in models:

        y_pred = temp[
            model
        ].to_numpy(dtype=float)

        overall_results.append({
            "Zmienna": variable,
            "Model": model,
            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "sMAPE (%)": smape(y_true, y_pred),
            "WAPE (%)": wape(y_true, y_pred)
        })


overall_df = pd.DataFrame(
    overall_results
)

overall_df = overall_df.sort_values(
    ["Zmienna", "RMSE"]
)

print("\nWYNIKI OGÓLNE:")
print(overall_df)

# 6. WYNIKI OSOBNO DLA KAŻDEJ SPÓŁKI

company_results = []

for variable in pred["Zmienna"].unique():

    temp_variable = pred[
        pred["Zmienna"] == variable
    ]

    for company in temp_variable["Spółka"].unique():

        temp = temp_variable[
            temp_variable["Spółka"] == company
        ]

        y_true = temp[
            "Wartość rzeczywista"
        ].to_numpy(dtype=float)

        for model in models:

            y_pred = temp[
                model
            ].to_numpy(dtype=float)

            company_results.append({
                "Spółka": company,
                "Zmienna": variable,
                "Model": model,
                "Liczba prognoz": len(temp),
                "MAE": mae(y_true, y_pred),
                "RMSE": rmse(y_true, y_pred),
                "sMAPE (%)": smape(y_true, y_pred),
                "WAPE (%)": wape(y_true, y_pred)
            })


company_df = pd.DataFrame(
    company_results
)

company_df = company_df.sort_values(
    [
        "Zmienna",
        "Spółka",
        "RMSE"
    ]
)


# 7. NAJLEPSZY MODEL DLA KAŻDEJ SPÓŁKI I ZMIENNEJ

best_company = (
    company_df
    .sort_values(
        [
            "Spółka",
            "Zmienna",
            "RMSE"
        ]
    )
    .groupby(
        [
            "Spółka",
            "Zmienna"
        ],
        as_index=False
    )
    .first()
)

# 8. ILE RAZY DANY MODEL BYŁ NAJLEPSZY

wins = (
    best_company
    .groupby(
        [
            "Zmienna",
            "Model"
        ]
    )
    .size()
    .reset_index(
        name="Liczba zwycięstw"
    )
)

# 9. MEDIANA METRYK MIĘDZY SPÓŁKAMI

median_company = (
    company_df
    .groupby(
        [
            "Zmienna",
            "Model"
        ],
        as_index=False
    )
    .agg({
        "MAE": "median",
        "RMSE": "median",
        "sMAPE (%)": "median",
        "WAPE (%)": "median"
    })
)

median_company = median_company.rename(
    columns={
        "MAE": "Mediana MAE",
        "RMSE": "Mediana RMSE",
        "sMAPE (%)": "Mediana sMAPE (%)",
        "WAPE (%)": "Mediana WAPE (%)"
    }
)


# 10. ZAPIS DO EXCELA


with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    overall_df.to_excel(
        writer,
        sheet_name="Wyniki ogólne",
        index=False
    )

    company_df.to_excel(
        writer,
        sheet_name="Wyniki per spółka",
        index=False
    )

    best_company.to_excel(
        writer,
        sheet_name="Najlepszy per spółka",
        index=False
    )

    wins.to_excel(
        writer,
        sheet_name="Liczba zwycięstw",
        index=False
    )

    median_company.to_excel(
        writer,
        sheet_name="Mediany spółek",
        index=False
    )

print("\nGOTOWE")
print("Utworzono plik:")
print(os.path.abspath(OUTPUT_FILE))


In [ ]:
# Finalny eksperyment LLM-1 (dane liczbowe)

import os
import re
import time
import requests
import pandas as pd
import numpy as np
# 1. USTAWIENIA

SOURCE_FILE = "dane_finansowe_gpw_2015_2024.xlsx"

PROMPTS_FILE = "dane_wejsciowe_LLM_liczby_v3.xlsx"

RESULTS_FILE = "wyniki_LLM_liczby_v3.xlsx"

SOURCE_SHEET = "Dane kwartalne"

# BEZPIECZNE POBRANIE KLUCZA OPENAI API
def get_openai_api_key():
    """
    Pobiera klucz z OPENAI_API_KEY. Jeżeli zmienna środowiskowa
    nie została wcześniej ustawiona, prosi o wpisanie klucza
    w sposób niewidoczny w Jupyterze i zapisuje go wyłącznie
    w pamięci bieżącej sesji Pythona.
    """
    key = os.getenv("OPENAI_API_KEY")

    if key:
        return key

    from getpass import getpass

    print("\nNie znaleziono zmiennej OPENAI_API_KEY.")
    print("Klucz zostanie użyty tylko w bieżącej sesji i nie będzie zapisany w pliku.")

    key = getpass("Wpisz OPENAI_API_KEY: ").strip()

    if not key:
        raise RuntimeError(
            "Nie podano klucza API. Ustaw OPENAI_API_KEY lub uruchom skrypt ponownie i wpisz klucz."
        )

    os.environ["OPENAI_API_KEY"] = key
    return key

MODEL = "gpt-5.6-luna"

URL = "https://api.openai.com/v1/responses"

HEADERS = {
    "Authorization": f"Bearer {get_openai_api_key()}",
    "Content-Type": "application/json"
}

HISTORY_LENGTH = 8

TEST_YEARS = [2023, 2024]


# 2. WCZYTANIE DANYCH ŹRÓDŁOWYCH

df = pd.read_excel(
    SOURCE_FILE,
    sheet_name=SOURCE_SHEET
)

df["Rok"] = pd.to_numeric(
    df["Rok"],
    errors="coerce"
)

df["Data publikacji"] = pd.to_datetime(
    df["Data publikacji"],
    errors="coerce"
)

df["Numer_kwartału"] = (
    df["Kwartał"]
    .astype(str)
    .str.extract(r"(\d)")
    .astype(int)
)

df = df.sort_values(
    [
        "Spółka",
        "Rok",
        "Numer_kwartału"
    ]
).reset_index(drop=True)

print("Liczba rekordów źródłowych:", len(df))


# 3. ANONIMIZACJA SPÓŁEK

companies = sorted(
    df["Spółka"].dropna().unique()
)

company_codes = {
    company: f"Spółka_{chr(65+i)}"
    for i, company in enumerate(companies)
}

print("\nMapowanie spółek:")

for company, code in company_codes.items():
    print(company, "->", code)


. FORMATOWANIE LICZB

def format_value(x):

    if pd.isna(x):
        return "brak"

    return str(
        int(
            round(float(x))
        )
    )


# 5. GENEROWANIE 80 PROMPTÓW

records = []

for company in df["Spółka"].unique():

    company_df = (
        df[
            df["Spółka"] == company
        ]
        .sort_values(
            [
                "Rok",
                "Numer_kwartału"
            ]
        )
        .reset_index(drop=True)
    )

    anonymous_name = company_codes[company]

    for i in range(len(company_df)):

        target = company_df.iloc[i]

        if target["Rok"] not in TEST_YEARS:
            continue

        if i < HISTORY_LENGTH:
            continue

        history = company_df.iloc[
            i-HISTORY_LENGTH:i
        ].copy()

        history_lines = []

        for j, (_, row) in enumerate(
            history.iterrows(),
            start=1
        ):

            relative_period = (
                f"T-{HISTORY_LENGTH-j+1}"
            )

            line = (
                f"{relative_period}: "
                f"PRZYCHODY={format_value(row['Przychody ze sprzedaży'])}; "
                f"EBIT={format_value(row['EBIT'])}; "
                f"ZYSK_NETTO={format_value(row['Zysk netto'])}"
            )

            history_lines.append(line)

        history_text = "\n".join(
            history_lines
        )

        prompt = f"""
Jesteś analitykiem finansowym wykonującym prognozę wyników przedsiębiorstwa.

Na podstawie WYŁĄCZNIE przedstawionych poniżej danych historycznych
oszacuj wyniki finansowe przedsiębiorstwa dla kolejnego kwartału.

Przedsiębiorstwo: {anonymous_name}

Dane obejmują osiem kolejnych kwartałów poprzedzających okres prognozy.

WAŻNE ZASADY DOTYCZĄCE JEDNOSTEK I SKALI:

Wszystkie wartości wejściowe są podane w tysiącach PLN.
Wszystkie prognozy również MUSZĄ być zapisane w tysiącach PLN.

Nie przeliczaj wartości na PLN, mln PLN ani mld PLN.
Nie skracaj liczb i nie zmieniaj ich jednostki.

Liczby wejściowe zapisano bez separatorów tysięcy.

Przykład:
wartość 8172000 oznacza 8 172 000 tys. PLN.

Jeżeli prognozujesz wartość 8500000 tys. PLN, zwróć:
PRZYCHODY: 8500000

NIE zwracaj:
8500
8500 mln
8.5
8,5 mld

Przed udzieleniem odpowiedzi sprawdź,
czy każda prognozowana wartość pozostaje zapisana
w tej samej jednostce i skali liczbowej co dane historyczne
dotyczące tej zmiennej.

Nie zmieniaj skali liczby tylko dlatego, że wartość jest duża.

DANE HISTORYCZNE:

{history_text}

Oszacuj dla kolejnego kwartału:

1. Przychody ze sprzedaży
2. EBIT
3. Zysk netto

Nie korzystaj z wiedzy o rzeczywistych spółkach ani z informacji
spoza danych przedstawionych w tym poleceniu.

Zwróć WYŁĄCZNIE:

PRZYCHODY: [pełna liczba całkowita w tys. PLN]
EBIT: [pełna liczba całkowita w tys. PLN]
ZYSK_NETTO: [pełna liczba całkowita w tys. PLN]

Nie dodawaj komentarza, uzasadnienia ani dodatkowego tekstu.
""".strip()

        records.append({

            "Spółka rzeczywista":
                company,

            "Spółka anonimowa":
                anonymous_name,

            "Rok prognozy":
                int(target["Rok"]),

            "Kwartał prognozy":
                target["Kwartał"],

            "Przychody rzeczywiste":
                target["Przychody ze sprzedaży"],

            "EBIT rzeczywisty":
                target["EBIT"],

            "Zysk netto rzeczywisty":
                target["Zysk netto"],

            "Prompt":
                prompt,

            "LLM Przychody":
                np.nan,

            "LLM EBIT":
                np.nan,

            "LLM Zysk netto":
                np.nan,

            "Odpowiedź surowa":
                ""
        })


llm_df = pd.DataFrame(
    records
)

print(
    "\nLiczba przygotowanych promptów:",
    len(llm_df)
)

print("\nLiczba prognoz na spółkę:")

print(
    llm_df
    .groupby("Spółka rzeczywista")
    .size()
)

# 6. ARKUSZ MAPOWANIA

mapping_df = pd.DataFrame(
    [
        {
            "Spółka rzeczywista":
                company,

            "Kod anonimowy":
                code
        }

        for company, code
        in company_codes.items()
    ]
)


# 7. ARKUSZ METODYKA

methodology_df = pd.DataFrame({

    "Element": [
        "Model",
        "Okres testowy",
        "Liczba spółek",
        "Horyzont prognozy",
        "Historia wejściowa",
        "Anonimizacja",
        "Zmienne prognozowane",
        "Jednostka",
        "Liczba promptów",
        "Wersja promptu"
    ],

    "Opis": [
        MODEL,
        "2023-2024",
        str(len(companies)),
        "1 kwartał naprzód",
        "8 wcześniejszych kwartałów",
        "Nazwy spółek zastąpiono kodami anonimowymi.",
        "Przychody ze sprzedaży, EBIT, zysk netto",
        "tys. PLN",
        str(len(llm_df)),
        "v3 - liczby bez separatorów tysięcy oraz "
        "jednoznaczna kontrola jednostki i skali"
    ]
})

# 8. ZAPIS PROMPTÓW PRZED WYSŁANIEM

with pd.ExcelWriter(
    PROMPTS_FILE,
    engine="openpyxl"
) as writer:

    llm_df.to_excel(
        writer,
        sheet_name="Prompty LLM",
        index=False
    )

    mapping_df.to_excel(
        writer,
        sheet_name="Mapowanie spółek",
        index=False
    )

    methodology_df.to_excel(
        writer,
        sheet_name="Metodyka",
        index=False
    )

print(
    "\nZapisano prompty:"
)

print(
    os.path.abspath(
        PROMPTS_FILE
    )
)

# 9. FUNKCJA WYCIĄGAJĄCA TEKST Z ODPOWIEDZI API

def extract_output_text(result):

    text = ""

    for item in result.get(
        "output",
        []
    ):

        if item.get("type") == "message":

            for content in item.get(
                "content",
                []
            ):

                if (
                    content.get("type")
                    == "output_text"
                ):

                    text += content.get(
                        "text",
                        ""
                    )

    return text.strip()

# 10. PARSER LICZB

def parse_number(text):

    if text is None:
        return np.nan

    text = str(text).strip()

    text = (
        text
        .replace("\xa0", "")
        .replace(" ", "")
        .replace(",", "")
    )

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        text
    )

    if not match:
        return np.nan

    try:
        return float(
            match.group()
        )

    except:
        return np.nan

# 11. PARSER ODPOWIEDZI

def parse_response(text):

    revenue_match = re.search(
        r"PRZYCHODY\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    ebit_match = re.search(
        r"EBIT\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    net_match = re.search(
        r"ZYSK_NETTO\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    revenue = (
        parse_number(
            revenue_match.group(1)
        )
        if revenue_match
        else np.nan
    )

    ebit = (
        parse_number(
            ebit_match.group(1)
        )
        if ebit_match
        else np.nan
    )

    net_income = (
        parse_number(
            net_match.group(1)
        )
        if net_match
        else np.nan
    )

    return (
        revenue,
        ebit,
        net_income
    )


# 12. WŁAŚCIWY PRZEBIEG 80 PROGNOZ

for i in range(
    len(llm_df)
):

    prompt = llm_df.loc[
        i,
        "Prompt"
    ]

    company = llm_df.loc[
        i,
        "Spółka anonimowa"
    ]

    year = llm_df.loc[
        i,
        "Rok prognozy"
    ]

    quarter = llm_df.loc[
        i,
        "Kwartał prognozy"
    ]

    print(
        f"\n{i+1}/{len(llm_df)} | "
        f"{company} | "
        f"{year} {quarter}"
    )

    payload = {
        "model":
            MODEL,

        "input":
            prompt
    }

    try:

        response = requests.post(
            URL,
            headers=HEADERS,
            json=payload,
            timeout=120
        )

        print(
            "Status:",
            response.status_code
        )

        if response.status_code != 200:

            error_text = (
                response.text[:1000]
            )

            print(
                "BŁĄD API:",
                error_text
            )

            llm_df.loc[
                i,
                "Odpowiedź surowa"
            ] = (
                "API_ERROR: "
                + error_text
            )

        else:

            result = response.json()

            raw_text = (
                extract_output_text(
                    result
                )
            )

            (
                revenue,
                ebit,
                net_income

            ) = parse_response(
                raw_text
            )

            llm_df.loc[
                i,
                "LLM Przychody"
            ] = revenue

            llm_df.loc[
                i,
                "LLM EBIT"
            ] = ebit

            llm_df.loc[
                i,
                "LLM Zysk netto"
            ] = net_income

            llm_df.loc[
                i,
                "Odpowiedź surowa"
            ] = raw_text

            print(
                "Przychody:",
                revenue,
                "| EBIT:",
                ebit,
                "| Zysk netto:",
                net_income
            )

    except Exception as e:

        print(
            "WYJĄTEK:",
            e
        )

        llm_df.loc[
            i,
            "Odpowiedź surowa"
        ] = (
            "EXCEPTION: "
            + str(e)
        )

    # ZAPIS PO KAŻDYM WYWOŁANIU

    llm_df.to_excel(
        RESULTS_FILE,
        sheet_name="Wyniki LLM",
        index=False
    )

    time.sleep(0.5)


# 13. KONTROLA KOŃCOWA

print("\n")
print("=" * 60)
print("KONTROLA KOŃCOWA")
print("=" * 60)

print(
    "Liczba obserwacji:",
    len(llm_df)
)

print("\nBraki danych:")

print(
    llm_df[
        [
            "LLM Przychody",
            "LLM EBIT",
            "LLM Zysk netto"
        ]
    ]
    .isna()
    .sum()
)

print("\nBłędy API / wyjątki:")

error_count = (
    llm_df[
        "Odpowiedź surowa"
    ]
    .astype(str)
    .str.contains(
        "API_ERROR|EXCEPTION",
        regex=True
    )
    .sum()
)

print(
    error_count
)

# 14. KONTROLA SKALI

def safe_ratio(
    pred,
    actual
):

    actual = actual.replace(
        0,
        np.nan
    )

    return (
        np.abs(pred)
        / np.abs(actual)
    )


llm_df[
    "Ratio_Przychody"
] = safe_ratio(

    llm_df[
        "LLM Przychody"
    ],

    llm_df[
        "Przychody rzeczywiste"
    ]
)


llm_df[
    "Ratio_EBIT"
] = safe_ratio(

    llm_df[
        "LLM EBIT"
    ],

    llm_df[
        "EBIT rzeczywisty"
    ]
)


llm_df[
    "Ratio_Zysk"
] = safe_ratio(

    llm_df[
        "LLM Zysk netto"
    ],

    llm_df[
        "Zysk netto rzeczywisty"
    ]
)


podejrzane = llm_df[
    (
        llm_df[
            "Ratio_Przychody"
        ] < 0.1
    )
    |
    (
        llm_df[
            "Ratio_Przychody"
        ] > 10
    )
    |
    (
        llm_df[
            "Ratio_EBIT"
        ] < 0.1
    )
    |
    (
        llm_df[
            "Ratio_EBIT"
        ] > 10
    )
    |
    (
        llm_df[
            "Ratio_Zysk"
        ] < 0.1
    )
    |
    (
        llm_df[
            "Ratio_Zysk"
        ] > 10
    )
].copy()


print(
    "\nLiczba obserwacji oznaczonych "
    "do dodatkowej kontroli:",
    len(podejrzane)
)

# 15. ZAPIS WERSJI Z KONTROLĄ SKALI

with pd.ExcelWriter(
    RESULTS_FILE,
    engine="openpyxl"
) as writer:

    llm_df.to_excel(
        writer,
        sheet_name="Wyniki LLM",
        index=False
    )

    podejrzane.to_excel(
        writer,
        sheet_name="Kontrola skali",
        index=False
    )

    methodology_df.to_excel(
        writer,
        sheet_name="Metodyka",
        index=False
    )


print("\n")
print("=" * 60)
print("GOTOWE")
print("=" * 60)

print(
    "Plik wynikowy:"
)

print(
    os.path.abspath(
        RESULTS_FILE
    )
)


In [ ]:
# ORLEN – pobranie i ekstrakcja oficjalnych materiałów
import os
import re
import time
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
import pymupdf


# 1. USTAWIENIA

ARCHIVE_URL = (
    "https://www.orlen.pl/pl/relacje-inwestorskie/"
    "prezentacje/prezentacje-wynikowe"
)

OUTPUT_FOLDER = "ORLEN_teksty"
OUTPUT_EXCEL = "ORLEN_teksty_LLM_2023_2024.xlsx"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/125 Safari/537.36"
    )
}


# 2. MAPOWANIE:

documents = [
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2022 Q4",
        "Fraza": "4. kwartał 2022"
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2023 Q1",
        "Fraza": "1. kwartał 2023"
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2023 Q2 / H1",
        "Fraza": "2. kwartał 2023"
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2023 Q3",
        "Fraza": "3. kwartał 2023"
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2023 Q4",
        "Fraza": "4. kwartał 2023"
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2024 Q1",
        "Fraza": "1. kwartał 2024"
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2024 Q2 / H1",
        "Fraza": "2. kwartał 2024"
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2024 Q3",
        "Fraza": "3. kwartał 2024"
    }
]

# 3. NORMALIZACJA TEKSTU

def normalize(text):

    if text is None:
        return ""

    text = (
        str(text)
        .lower()
        .replace("\xa0", " ")
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# 4. POBRANIE STRONY ARCHIWUM

response = requests.get(
    ARCHIVE_URL,
    headers=HEADERS,
    timeout=60
)

print(
    "Status strony ORLEN:",
    response.status_code
)

response.raise_for_status()

soup = BeautifulSoup(
    response.text,
    "html.parser"
)


# 5. ZEBRANIE WSZYSTKICH LINKÓW DO PDF

all_links = []

for a in soup.find_all(
    "a",
    href=True
):

    href = a.get(
        "href",
        ""
    )

    label = normalize(
        a.get_text(
            " ",
            strip=True
        )
    )

    full_url = urljoin(
        ARCHIVE_URL,
        href
    )

    if (
        ".pdf" in full_url.lower()
        or "coredownload" in full_url.lower()
    ):

        all_links.append({
            "tekst": label,
            "url": full_url
        })


print(
    "Znalezionych potencjalnych PDF:",
    len(all_links)
)

# 6. FUNKCJA SZUKAJĄCA ODPOWIEDNIEJ PREZENTACJI

def find_pdf(phrase):

    phrase_normalized = normalize(
        phrase
    )

    matches = []

    for item in all_links:

        combined = normalize(
            item["tekst"]
            + " "
            + item["url"]
        )

        if phrase_normalized in combined:
            matches.append(
                item
            )

    preferred = []

    for item in matches:

        combined = normalize(
            item["tekst"]
            + " "
            + item["url"]
        )

        if (
            "szacun" not in combined
            and "wprowadzenie" not in combined
        ):
            preferred.append(
                item
            )

    if preferred:
        return preferred[0]

    if matches:
        return matches[0]

    return None


# 7. EKSTRAKCJA TEKSTU Z PDF

def extract_pdf_text(
    pdf_path
):

    doc = pymupdf.open(
        pdf_path
    )

    pages = []

    for page_number, page in enumerate(
        doc,
        start=1
    ):

        page_text = page.get_text(
            "text"
        )

        page_text = re.sub(
            r"[ \t]+",
            " ",
            page_text
        )

        page_text = re.sub(
            r"\n{3,}",
            "\n\n",
            page_text
        )

        pages.append(
            page_text.strip()
        )

    doc.close()

    return "\n\n".join(
        pages
    )


# 8. POBIERANIE 8 DOKUMENTÓW

results = []

for index, item in enumerate(
    documents,
    start=1
):

    year = item[
        "Rok prognozy"
    ]

    quarter = item[
        "Kwartał prognozy"
    ]

    phrase = item[
        "Fraza"
    ]

    print()
    print(
        "=" * 65
    )
    print(
        f"{index}/8 | "
        f"Prognoza: {year} {quarter} | "
        f"Szukam: {phrase}"
    )

    found = find_pdf(
        phrase
    )

    if found is None:

        print(
            "NIE ZNALEZIONO PDF"
        )

        results.append({
            "Spółka": "ORLEN",
            "Rok prognozy": year,
            "Kwartał prognozy": quarter,
            "Okres dokumentu": item["Okres dokumentu"],
            "Fraza wyszukiwania": phrase,
            "Status": "NIE ZNALEZIONO",
            "URL": "",
            "Plik PDF": "",
            "Plik TXT": "",
            "Liczba znaków": 0,
            "Liczba słów": 0,
            "Tekst": ""
        })

        continue


    pdf_url = found[
        "url"
    ]

    print(
        "Znaleziono:"
    )

    print(
        pdf_url
    )

    file_base = (
        f"ORLEN_"
        f"{year}_"
        f"{quarter}_"
        f"source"
    )

    pdf_path = os.path.join(
        OUTPUT_FOLDER,
        file_base + ".pdf"
    )

    txt_path = os.path.join(
        OUTPUT_FOLDER,
        file_base + ".txt"
    )

    pdf_response = requests.get(
        pdf_url,
        headers=HEADERS,
        timeout=120
    )

    print(
        "Status PDF:",
        pdf_response.status_code
    )

    if pdf_response.status_code != 200:

        results.append({
            "Spółka": "ORLEN",
            "Rok prognozy": year,
            "Kwartał prognozy": quarter,
            "Okres dokumentu": item["Okres dokumentu"],
            "Fraza wyszukiwania": phrase,
            "Status": (
                f"BŁĄD HTTP "
                f"{pdf_response.status_code}"
            ),
            "URL": pdf_url,
            "Plik PDF": "",
            "Plik TXT": "",
            "Liczba znaków": 0,
            "Liczba słów": 0,
            "Tekst": ""
        })

        continue


    with open(
        pdf_path,
        "wb"
    ) as f:

        f.write(
            pdf_response.content
        )

    try:

        text = extract_pdf_text(
            pdf_path
        )

    except Exception as e:

        print(
            "BŁĄD ekstrakcji:",
            e
        )

        results.append({
            "Spółka": "ORLEN",
            "Rok prognozy": year,
            "Kwartał prognozy": quarter,
            "Okres dokumentu": item["Okres dokumentu"],
            "Fraza wyszukiwania": phrase,
            "Status": (
                "BŁĄD EKSTRAKCJI: "
                + str(e)
            ),
            "URL": pdf_url,
            "Plik PDF": pdf_path,
            "Plik TXT": "",
            "Liczba znaków": 0,
            "Liczba słów": 0,
            "Tekst": ""
        })

        continue

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            text
        )


    char_count = len(
        text
    )

    word_count = len(
        text.split()
    )


    print(
        "Liczba znaków:",
        char_count
    )

    print(
        "Liczba słów:",
        word_count
    )


    results.append({
        "Spółka": "ORLEN",
        "Rok prognozy": year,
        "Kwartał prognozy": quarter,
        "Okres dokumentu": item["Okres dokumentu"],
        "Fraza wyszukiwania": phrase,
        "Status": "OK",
        "URL": pdf_url,
        "Plik PDF": pdf_path,
        "Plik TXT": txt_path,
        "Liczba znaków": char_count,
        "Liczba słów": word_count,
        "Tekst": text
    })


    time.sleep(
        1
    )

# 9. DATAFRAME

orlen_texts = pd.DataFrame(
    results
)


print()
print(
    "=" * 65
)
print(
    "PODSUMOWANIE"
)
print(
    "=" * 65
)

print(
    orlen_texts[
        [
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "Status",
            "Liczba znaków",
            "Liczba słów"
        ]
    ]
)

# 10. ZAPIS DO EXCELA

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    orlen_texts.to_excel(
        writer,
        sheet_name="ORLEN teksty",
        index=False
    )


print()
print(
    "Gotowy Excel:"
)

print(
    os.path.abspath(
        OUTPUT_EXCEL
    )
)

print()
print(
    "Folder PDF/TXT:"
)

print(
    os.path.abspath(
        OUTPUT_FOLDER
    )
)


In [ ]:
# ORLEN – przygotowanie promptów LLM-2

import pandas as pd
import os

# ============================================================
# PLIKI
# ============================================================

PROMPTS_FILE = "dane_wejsciowe_LLM_liczby_v3.xlsx"
TEXT_FILE = "ORLEN_teksty_LLM_2023_2024.xlsx"

OUTPUT_FILE = "ORLEN_LLM_liczby_plus_tekst.xlsx"



# 1. WCZYTANIE PROMPTÓW LICZBOWYCH


prompts = pd.read_excel(
    PROMPTS_FILE,
    sheet_name="Prompty LLM"
)

# wybieramy tylko ORLEN
orlen_prompts = prompts[
    prompts["Spółka rzeczywista"] == "ORLEN"
].copy()

print("Promptów ORLEN:", len(orlen_prompts))


# 2. WCZYTANIE TEKSTÓW


texts = pd.read_excel(
    TEXT_FILE,
    sheet_name="ORLEN teksty"
)

print("Tekstów ORLEN:", len(texts))


# 3. POŁĄCZENIE PO ROKU I KWARTALE PROGNOZY

merged = pd.merge(
    orlen_prompts,
    texts[
        [
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "URL",
            "Liczba znaków",
            "Liczba słów",
            "Tekst"
        ]
    ],
    on=[
        "Rok prognozy",
        "Kwartał prognozy"
    ],
    how="left"
)

print("\nLiczba połączonych rekordów:", len(merged))

print("\nBrakujące teksty:")
print(merged["Tekst"].isna().sum())

# 4. FUNKCJA CZYSZCZĄCA TEKST

def clean_text(text):

    if pd.isna(text):
        return ""

    text = str(text)

    # usunięcie nadmiarowych pustych linii
    lines = [
        line.strip()
        for line in text.splitlines()
        if line.strip()
    ]

    return "\n".join(lines)


merged["Tekst oczyszczony"] = (
    merged["Tekst"]
    .apply(clean_text)
)


# 5. BUDOWA PROMPTU LLM-2

def build_text_prompt(row):

    numeric_prompt = row["Prompt"]

    report_text = row["Tekst oczyszczony"]

    source_period = row["Okres dokumentu"]

    new_prompt = f"""
{numeric_prompt}

DODATKOWY KONTEKST TEKSTOWY:

Poniżej przedstawiono treść materiału wynikowego dotyczącego
poprzedniego okresu sprawozdawczego ({source_period}).

Materiał ten był dostępny w momencie wykonywania prognozy.

Wykorzystaj informacje tekstowe jedynie jako dodatkowy kontekst
do danych liczbowych. Zwróć szczególną uwagę na informacje dotyczące:

- zmian przychodów,
- rentowności i marż,
- kosztów działalności,
- sytuacji poszczególnych segmentów,
- czynników jednorazowych,
- inwestycji,
- warunków rynkowych,
- ryzyk,
- oczekiwań i perspektyw kolejnych okresów.

Nie zakładaj, że każda informacja zawarta w tekście musi mieć
bezpośredni wpływ na kolejny kwartał.

TREŚĆ MATERIAŁU:

--- POCZĄTEK TEKSTU ---

{report_text}

--- KONIEC TEKSTU ---

Na podstawie danych liczbowych oraz przedstawionego kontekstu tekstowego
oszacuj wyniki kolejnego kwartału.

Zwróć WYŁĄCZNIE:

PRZYCHODY: [pełna liczba całkowita w tys. PLN]
EBIT: [pełna liczba całkowita w tys. PLN]
ZYSK_NETTO: [pełna liczba całkowita w tys. PLN]

Nie dodawaj komentarza, uzasadnienia ani dodatkowego tekstu.
""".strip()

    return new_prompt


merged["Prompt LLM-2"] = merged.apply(
    build_text_prompt,
    axis=1
)


# 6. KOLUMNY NA WYNIKI


merged["LLM2 Przychody"] = None
merged["LLM2 EBIT"] = None
merged["LLM2 Zysk netto"] = None
merged["Odpowiedź LLM2"] = ""


# 7. ZAPIS


merged.to_excel(
    OUTPUT_FILE,
    sheet_name="ORLEN LLM2",
    index=False
)

print("\nGOTOWE")
print(os.path.abspath(OUTPUT_FILE))


# ORLEN – wykonanie prognoz LLM-2

import os
import re
import time
import requests
import pandas as pd
import numpy as np


# USTAWIENIA

INPUT_FILE = "ORLEN_LLM_liczby_plus_tekst.xlsx"
OUTPUT_FILE = "ORLEN_wyniki_LLM2.xlsx"

MODEL = "gpt-5.6-luna"

URL = "https://api.openai.com/v1/responses"

HEADERS = {
    "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
    "Content-Type": "application/json"
}


# WCZYTANIE

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="ORLEN LLM2"
)

df["Odpowiedź LLM2"] = (
    df["Odpowiedź LLM2"]
    .fillna("")
    .astype(str)
)

# FUNKCJE

def extract_output_text(result):

    text = ""

    for item in result.get("output", []):

        if item.get("type") == "message":

            for content in item.get("content", []):

                if content.get("type") == "output_text":

                    text += content.get("text", "")

    return text.strip()


def parse_number(text):

    if text is None:
        return np.nan

    text = (
        str(text)
        .strip()
        .replace("\xa0", "")
        .replace(" ", "")
        .replace(",", "")
    )

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        text
    )

    if not match:
        return np.nan

    try:
        return float(match.group())
    except:
        return np.nan


def parse_response(text):

    revenue_match = re.search(
        r"PRZYCHODY\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    ebit_match = re.search(
        r"EBIT\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    net_match = re.search(
        r"ZYSK_NETTO\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    revenue = (
        parse_number(revenue_match.group(1))
        if revenue_match
        else np.nan
    )

    ebit = (
        parse_number(ebit_match.group(1))
        if ebit_match
        else np.nan
    )

    net_income = (
        parse_number(net_match.group(1))
        if net_match
        else np.nan
    )

    return revenue, ebit, net_income

# 8 PROGNOZ

for i in range(len(df)):

    prompt = df.loc[i, "Prompt LLM-2"]

    year = df.loc[i, "Rok prognozy"]
    quarter = df.loc[i, "Kwartał prognozy"]

    print(
        f"\n{i+1}/{len(df)} | "
        f"ORLEN | {year} {quarter}"
    )

    payload = {
        "model": MODEL,
        "input": prompt
    }

    try:

        response = requests.post(
            URL,
            headers=HEADERS,
            json=payload,
            timeout=180
        )

        print("Status:", response.status_code)

        if response.status_code != 200:

            print(response.text[:1000])

            df.loc[
                i,
                "Odpowiedź LLM2"
            ] = (
                "API_ERROR: "
                + response.text[:1000]
            )

            continue

        result = response.json()

        raw_text = extract_output_text(
            result
        )

        revenue, ebit, net_income = (
            parse_response(raw_text)
        )

        df.loc[
            i,
            "LLM2 Przychody"
        ] = revenue

        df.loc[
            i,
            "LLM2 EBIT"
        ] = ebit

        df.loc[
            i,
            "LLM2 Zysk netto"
        ] = net_income

        df.loc[
            i,
            "Odpowiedź LLM2"
        ] = raw_text

        print(
            "Przychody:",
            revenue,
            "| EBIT:",
            ebit,
            "| Zysk netto:",
            net_income
        )

    except Exception as e:

        print("BŁĄD:", e)

        df.loc[
            i,
            "Odpowiedź LLM2"
        ] = (
            "EXCEPTION: "
            + str(e)
        )

    # zapis po każdym zapytaniu
    df.to_excel(
        OUTPUT_FILE,
        sheet_name="ORLEN LLM2",
        index=False
    )

    time.sleep(0.5)


print("\nGOTOWE")
print(os.path.abspath(OUTPUT_FILE))


In [ ]:
# KGHM – pobranie i ekstrakcja oficjalnych materiałów

import os
import re
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import pymupdf

# 1. USTAWIENIA

OUTPUT_FOLDER = "KGHM_teksty"
OUTPUT_EXCEL = "KGHM_teksty_LLM_2023_2024.xlsx"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/125 Safari/537.36"
    )
}

# 2. STRONY PREZENTACJI

documents = [

    # prognoza 2023 Q1 -> wyniki Q4 2022
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2022 Q4",
        "Strona": (
            "https://kghm.com/pl/"
            "kghm-q4-2022-prezentacja-wynikow"
        )
    },

    # prognoza 2023 Q2 -> wyniki Q1 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2023 Q1",
        "Strona": (
            "https://kghm.com/pl/inwestorzy/centrum-wynikow"
        ),
        "Szukana fraza": "Q1 2023"
    },

    # prognoza 2023 Q3 -> H1 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2023 H1",
        "Strona": (
            "https://kghm.com/pl/"
            "grupa-kghm-h1-2023-prezentacja"
        )
    },

    # prognoza 2023 Q4 -> Q3 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2023 Q3",
        "Strona": (
            "https://kghm.com/pl/"
            "kghmq32023-prezentacja"
        )
    },

    # prognoza 2024 Q1 -> Q4 2023
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2023 Q4",
        "Strona": (
            "https://kghm.com/pl/"
            "kghm-q4-2023-prezentacja"
        )
    },

    # prognoza 2024 Q2 -> Q1 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2024 Q1",
        "Strona": (
            "https://kghm.com/pl/"
            "kghm-q1-2024-prezentacja"
        )
    },

    # prognoza 2024 Q3 -> H1 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2024 H1",
        "Strona": (
            "https://kghm.com/pl/"
            "kghmh12024prezentacja"
        )
    },

    # prognoza 2024 Q4 -> Q3 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2024 Q3",
        "Strona": (
            "https://kghm.com/pl/inwestorzy/centrum-wynikow"
        ),
        "Szukana fraza": "Q3 2024"
    }
]

# 3. FUNKCJA CZYSZCZĄCA TEKST

def normalize(text):

    if text is None:
        return ""

    text = str(text).replace("\xa0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()

# 4. ZNAJDOWANIE LINKU PDF NA STRONIE


def find_pdf_on_page(page_url, search_phrase=None):

    response = requests.get(
        page_url,
        headers=HEADERS,
        timeout=60
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    candidates = []

    for a in soup.find_all("a", href=True):

        href = a.get("href", "")

        full_url = urljoin(
            page_url,
            href
        )

        surrounding = normalize(
            a.parent.get_text(
                " ",
                strip=True
            )
        )

        label = normalize(
            a.get_text(
                " ",
                strip=True
            )
        )

        combined = (
            label
            + " "
            + surrounding
            + " "
            + full_url
        ).lower()

        if ".pdf" not in full_url.lower():
            continue

        if search_phrase:

            words = (
                search_phrase
                .lower()
                .split()
            )

            score = sum(
                word in combined
                for word in words
            )

        else:

            score = 1

        candidates.append(
            (
                score,
                full_url,
                combined
            )
        )

    if not candidates:
        return None

    candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    return candidates[0][1]

# 5. EKSTRAKCJA TEKSTU Z PDF

def extract_pdf_text(pdf_path):

    doc = pymupdf.open(
        pdf_path
    )

    pages = []

    for page in doc:

        page_text = page.get_text(
            "text"
        )

        page_text = re.sub(
            r"[ \t]+",
            " ",
            page_text
        )

        page_text = re.sub(
            r"\n{3,}",
            "\n\n",
            page_text
        )

        pages.append(
            page_text.strip()
        )

    doc.close()

    return "\n\n".join(
        pages
    )

# 6. POBIERANIE DOKUMENTÓW

results = []

for i, item in enumerate(
    documents,
    start=1
):

    year = item["Rok prognozy"]
    quarter = item["Kwartał prognozy"]
    source_period = item["Okres dokumentu"]

    print()
    print("=" * 70)

    print(
        f"{i}/8 | KGHM | "
        f"prognoza {year} {quarter} | "
        f"źródło {source_period}"
    )

    try:

        pdf_url = find_pdf_on_page(
            item["Strona"],
            item.get("Szukana fraza")
        )

        if pdf_url is None:

            raise Exception(
                "Nie znaleziono PDF na stronie"
            )

        print("PDF:")
        print(pdf_url)

        filename = (
            f"KGHM_{year}_{quarter}_source"
        )

        pdf_path = os.path.join(
            OUTPUT_FOLDER,
            filename + ".pdf"
        )

        txt_path = os.path.join(
            OUTPUT_FOLDER,
            filename + ".txt"
        )


        response = requests.get(
            pdf_url,
            headers=HEADERS,
            timeout=120
        )

        print(
            "Status PDF:",
            response.status_code
        )

        response.raise_for_status()

        with open(
            pdf_path,
            "wb"
        ) as f:

            f.write(
                response.content
            )


        text = extract_pdf_text(
            pdf_path
        )

        with open(
            txt_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(text)

        char_count = len(text)
        word_count = len(text.split())

        print(
            "Znaki:",
            char_count
        )

        print(
            "Słowa:",
            word_count
        )

        results.append({

            "Spółka":
                "KGHM Polska Miedź",

            "Rok prognozy":
                year,

            "Kwartał prognozy":
                quarter,

            "Okres dokumentu":
                source_period,

            "Status":
                "OK",

            "Strona źródłowa":
                item["Strona"],

            "URL":
                pdf_url,

            "Plik PDF":
                pdf_path,

            "Plik TXT":
                txt_path,

            "Liczba znaków":
                char_count,

            "Liczba słów":
                word_count,

            "Tekst":
                text
        })

    except Exception as e:

        print(
            "BŁĄD:",
            e
        )

        results.append({

            "Spółka":
                "KGHM Polska Miedź",

            "Rok prognozy":
                year,

            "Kwartał prognozy":
                quarter,

            "Okres dokumentu":
                source_period,

            "Status":
                "BŁĄD: " + str(e),

            "Strona źródłowa":
                item["Strona"],

            "URL":
                "",

            "Plik PDF":
                "",

            "Plik TXT":
                "",

            "Liczba znaków":
                0,

            "Liczba słów":
                0,

            "Tekst":
                ""
        })

    time.sleep(1)

# 7. PODSUMOWANIE

kghm_texts = pd.DataFrame(
    results
)

print()
print("=" * 70)
print("PODSUMOWANIE KGHM")
print("=" * 70)

display(
    kghm_texts[
        [
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "Status",
            "Liczba znaków",
            "Liczba słów"
        ]
    ]
)

# 8. ZAPIS

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    kghm_texts.to_excel(
        writer,
        sheet_name="KGHM teksty",
        index=False
    )


print("\nGotowy Excel:")
print(
    os.path.abspath(
        OUTPUT_EXCEL
    )
)

print("\nFolder PDF/TXT:")
print(
    os.path.abspath(
        OUTPUT_FOLDER
    )
)



In [ ]:
# KGHM – przygotowanie i wykonanie prognoz LLM-2

import os
import re
import time
import requests
import pandas as pd
import numpy as np

# 1. USTAWIENIA

NUMERIC_PROMPTS_FILE = "dane_wejsciowe_LLM_liczby_v3.xlsx"
TEXT_FILE = "KGHM_teksty_LLM_2023_2024.xlsx"
LLM1_FILE = "wyniki_LLM_liczby_v3.xlsx"

OUTPUT_RESULTS = "KGHM_wyniki_LLM2.xlsx"
OUTPUT_COMPARISON = "KGHM_porownanie_LLM1_LLM2.xlsx"

MODEL = "gpt-5.6-luna"

URL = "https://api.openai.com/v1/responses"

HEADERS = {
    "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
    "Content-Type": "application/json"
}

# 2. WCZYTANIE PROMPTÓW LICZBOWYCH

prompts = pd.read_excel(
    NUMERIC_PROMPTS_FILE,
    sheet_name="Prompty LLM"
)

kghm_prompts = prompts[
    prompts["Spółka rzeczywista"] == "KGHM Polska Miedź"
].copy()

print("Promptów KGHM:", len(kghm_prompts))


# 3. WCZYTANIE TEKSTÓW KGHM

texts = pd.read_excel(
    TEXT_FILE,
    sheet_name="KGHM teksty"
)

print("Tekstów KGHM:", len(texts))


# 4. POŁĄCZENIE

merged = pd.merge(
    kghm_prompts,
    texts[
        [
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "URL",
            "Liczba znaków",
            "Liczba słów",
            "Tekst"
        ]
    ],
    on=[
        "Rok prognozy",
        "Kwartał prognozy"
    ],
    how="left"
)

print("Połączonych obserwacji:", len(merged))
print("Brakujące teksty:", merged["Tekst"].isna().sum())


# 5. CZYSZCZENIE TEKSTU

def clean_text(text):

    if pd.isna(text):
        return ""

    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    return "\n".join(lines)


merged["Tekst oczyszczony"] = (
    merged["Tekst"]
    .apply(clean_text)
)


# 6. FINALNY PROMPT LLM-2

def build_llm2_prompt(row):

    numeric_prompt = row["Prompt"]
    report_text = row["Tekst oczyszczony"]
    source_period = row["Okres dokumentu"]

    return f"""
{numeric_prompt}

DODATKOWY KONTEKST TEKSTOWY:

Poniżej przedstawiono treść materiału wynikowego dotyczącego
poprzedniego okresu sprawozdawczego ({source_period}).

Materiał ten był dostępny w momencie wykonywania prognozy.

Wykorzystaj informacje tekstowe jedynie jako dodatkowy kontekst
do danych liczbowych. Zwróć szczególną uwagę na informacje dotyczące:

- zmian przychodów,
- rentowności i marż,
- kosztów działalności,
- sytuacji poszczególnych segmentów,
- czynników jednorazowych,
- inwestycji,
- warunków rynkowych,
- ryzyk,
- oczekiwań i perspektyw kolejnych okresów.

Nie zakładaj, że każda informacja zawarta w tekście musi mieć
bezpośredni wpływ na kolejny kwartał.

TREŚĆ MATERIAŁU:

--- POCZĄTEK TEKSTU ---

{report_text}

--- KONIEC TEKSTU ---

Na podstawie danych liczbowych oraz przedstawionego kontekstu tekstowego
oszacuj wyniki kolejnego kwartału.

Zwróć WYŁĄCZNIE:

PRZYCHODY: [pełna liczba całkowita w tys. PLN]
EBIT: [pełna liczba całkowita w tys. PLN]
ZYSK_NETTO: [pełna liczba całkowita w tys. PLN]

Nie dodawaj komentarza, uzasadnienia ani dodatkowego tekstu.
""".strip()


merged["Prompt LLM-2"] = merged.apply(
    build_llm2_prompt,
    axis=1
)

merged["LLM2 Przychody"] = np.nan
merged["LLM2 EBIT"] = np.nan
merged["LLM2 Zysk netto"] = np.nan
merged["Odpowiedź LLM2"] = ""

# 7. FUNKCJE API / PARSER

def extract_output_text(result):

    text = ""

    for item in result.get("output", []):

        if item.get("type") == "message":

            for content in item.get("content", []):

                if content.get("type") == "output_text":
                    text += content.get("text", "")

    return text.strip()


def parse_number(text):

    if text is None:
        return np.nan

    text = (
        str(text)
        .strip()
        .replace("\xa0", "")
        .replace(" ", "")
        .replace(",", "")
    )

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        text
    )

    if not match:
        return np.nan

    return float(match.group())


def parse_response(text):

    revenue_match = re.search(
        r"PRZYCHODY\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    ebit_match = re.search(
        r"EBIT\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    net_match = re.search(
        r"ZYSK_NETTO\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    revenue = (
        parse_number(revenue_match.group(1))
        if revenue_match else np.nan
    )

    ebit = (
        parse_number(ebit_match.group(1))
        if ebit_match else np.nan
    )

    net_income = (
        parse_number(net_match.group(1))
        if net_match else np.nan
    )

    return revenue, ebit, net_income

# 8. WYKONANIE 8 PROGNOZ KGHM

for i in range(len(merged)):

    prompt = merged.loc[i, "Prompt LLM-2"]

    year = merged.loc[i, "Rok prognozy"]
    quarter = merged.loc[i, "Kwartał prognozy"]

    print(
        f"\n{i+1}/{len(merged)} | "
        f"KGHM | {year} {quarter}"
    )

    payload = {
        "model": MODEL,
        "input": prompt
    }

    try:

        response = requests.post(
            URL,
            headers=HEADERS,
            json=payload,
            timeout=180
        )

        print("Status:", response.status_code)

        if response.status_code != 200:

            print(response.text[:1000])

            merged.loc[
                i,
                "Odpowiedź LLM2"
            ] = (
                "API_ERROR: "
                + response.text[:1000]
            )

        else:

            result = response.json()

            raw_text = extract_output_text(result)

            revenue, ebit, net_income = (
                parse_response(raw_text)
            )

            merged.loc[i, "LLM2 Przychody"] = revenue
            merged.loc[i, "LLM2 EBIT"] = ebit
            merged.loc[i, "LLM2 Zysk netto"] = net_income
            merged.loc[i, "Odpowiedź LLM2"] = raw_text

            print(
                "Przychody:", revenue,
                "| EBIT:", ebit,
                "| Zysk netto:", net_income
            )

    except Exception as e:

        print("BŁĄD:", e)

        merged.loc[
            i,
            "Odpowiedź LLM2"
        ] = "EXCEPTION: " + str(e)

    # zapis po każdym zapytaniu
    merged.to_excel(
        OUTPUT_RESULTS,
        sheet_name="KGHM LLM2",
        index=False
    )

    time.sleep(0.5)

# 9. WCZYTANIE LLM-1 DLA KGHM

llm1 = pd.read_excel(
    LLM1_FILE,
    sheet_name="Wyniki LLM"
)

llm1 = llm1[
    llm1["Spółka rzeczywista"] == "KGHM Polska Miedź"
].copy()

llm1 = llm1[
    [
        "Rok prognozy",
        "Kwartał prognozy",
        "Przychody rzeczywiste",
        "EBIT rzeczywisty",
        "Zysk netto rzeczywisty",
        "LLM Przychody",
        "LLM EBIT",
        "LLM Zysk netto"
    ]
].rename(
    columns={
        "LLM Przychody": "LLM1 Przychody",
        "LLM EBIT": "LLM1 EBIT",
        "LLM Zysk netto": "LLM1 Zysk netto"
    }
)


# 10. POŁĄCZENIE LLM-1 i LLM-2

comparison = pd.merge(
    llm1,
    merged[
        [
            "Rok prognozy",
            "Kwartał prognozy",
            "LLM2 Przychody",
            "LLM2 EBIT",
            "LLM2 Zysk netto"
        ]
    ],
    on=[
        "Rok prognozy",
        "Kwartał prognozy"
    ]
)

# 11. METRYKI

def mae(y_true, y_pred):
    return np.mean(
        np.abs(y_true - y_pred)
    )


def rmse(y_true, y_pred):
    return np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )


def smape(y_true, y_pred):

    denominator = (
        np.abs(y_true)
        + np.abs(y_pred)
    )

    mask = denominator != 0

    return (
        np.mean(
            2
            * np.abs(y_pred[mask] - y_true[mask])
            / denominator[mask]
        )
        * 100
    )


def wape(y_true, y_pred):

    denominator = np.sum(
        np.abs(y_true)
    )

    if denominator == 0:
        return np.nan

    return (
        np.sum(
            np.abs(y_true - y_pred)
        )
        / denominator
        * 100
    )


variables = {

    "Przychody": {
        "actual": "Przychody rzeczywiste",
        "LLM1": "LLM1 Przychody",
        "LLM2": "LLM2 Przychody"
    },

    "EBIT": {
        "actual": "EBIT rzeczywisty",
        "LLM1": "LLM1 EBIT",
        "LLM2": "LLM2 EBIT"
    },

    "Zysk_netto": {
        "actual": "Zysk netto rzeczywisty",
        "LLM1": "LLM1 Zysk netto",
        "LLM2": "LLM2 Zysk netto"
    }
}


results = []

for variable, cols in variables.items():

    y_true = comparison[
        cols["actual"]
    ].to_numpy(dtype=float)

    for model in ["LLM1", "LLM2"]:

        y_pred = comparison[
            cols[model]
        ].to_numpy(dtype=float)

        results.append({
            "Zmienna": variable,

            "Model": (
                "LLM - liczby"
                if model == "LLM1"
                else "LLM - liczby + tekst"
            ),

            "MAE": mae(y_true, y_pred),
            "RMSE": rmse(y_true, y_pred),
            "sMAPE (%)": smape(y_true, y_pred),
            "WAPE (%)": wape(y_true, y_pred)
        })


results_df = pd.DataFrame(results)


# 12. POPRAWA RMSE

improvement = []

for variable in variables.keys():

    temp = results_df[
        results_df["Zmienna"] == variable
    ]

    rmse1 = temp[
        temp["Model"] == "LLM - liczby"
    ]["RMSE"].iloc[0]

    rmse2 = temp[
        temp["Model"] == "LLM - liczby + tekst"
    ]["RMSE"].iloc[0]

    change = (
        (rmse1 - rmse2)
        / rmse1
        * 100
    )

    improvement.append({
        "Zmienna": variable,
        "RMSE LLM-1": rmse1,
        "RMSE LLM-2": rmse2,
        "Poprawa RMSE po dodaniu tekstu (%)": change
    })


improvement_df = pd.DataFrame(
    improvement
)


print("\n======================================")
print("KGHM - LLM1 VS LLM2")
print("======================================")

print(results_df)

print("\n======================================")
print("WPŁYW DODANIA TEKSTU")
print("======================================")

print(improvement_df)

# 13. ZAPIS PORÓWNANIA

with pd.ExcelWriter(
    OUTPUT_COMPARISON,
    engine="openpyxl"
) as writer:

    comparison.to_excel(
        writer,
        sheet_name="Prognozy",
        index=False
    )

    results_df.to_excel(
        writer,
        sheet_name="Metryki",
        index=False
    )

    improvement_df.to_excel(
        writer,
        sheet_name="Wpływ tekstu",
        index=False
    )


print("\nGOTOWE")
print(OUTPUT_RESULTS)
print(OUTPUT_COMPARISON)


In [ ]:
# Pozostałe 8 spółek – zebranie materiałów tekstowych
# LLM-2

import os
import re
import time
import unicodedata
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
import pymupdf

# 1. USTAWIENIA

OUTPUT_FOLDER = "LLM2_materialy_8_spolek"
OUTPUT_EXCEL = "LLM2_materialy_8_spolek.xlsx"

os.makedirs(OUTPUT_FOLDER, exist_ok=True)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/125 Safari/537.36"
    )
}

TIMEOUT = 90

# 2. OFICJALNE STRONY ŹRÓDŁOWE

SOURCES = {

    "Asseco Poland": {
        2022: "https://inwestor.asseco.com/materialy-informacyjne/prezentacje/2022/",
        2023: "https://inwestor.asseco.com/materialy-informacyjne/prezentacje/2023/",
        2024: "https://inwestor.asseco.com/materialy-informacyjne/prezentacje/2024/",
    },

    "Budimex": {
        2022: "https://budimex.pl/relacje-inwestorskie/informacje-dla-inwestorow/prezentacje/",
        2023: "https://budimex.pl/relacje-inwestorskie/informacje-dla-inwestorow/prezentacje/",
        2024: "https://budimex.pl/relacje-inwestorskie/informacje-dla-inwestorow/prezentacje/",
    },

    "CD Projekt": {
        2022: "https://www.cdprojekt.com/pl/rok-obrotowy/2022/",
        2023: "https://www.cdprojekt.com/pl/rok-obrotowy/2023/",
        2024: "https://www.cdprojekt.com/pl/rok-obrotowy/2024/",
    },

    "Cyfrowy Polsat": {
        2022: "https://grupapolsatplus.pl/pl/relacje-inwestorskie/prezentacje",
        2023: "https://grupapolsatplus.pl/pl/relacje-inwestorskie/prezentacje",
        2024: "https://grupapolsatplus.pl/pl/relacje-inwestorskie/prezentacje",
    },

    "Grupa Kęty": {
        2022: "https://grupakety.com/relacje-inwestorskie/grupa-kety-na-gpw/raporty-okresowe-i-prezentacje/",
        2023: "https://grupakety.com/relacje-inwestorskie/grupa-kety-na-gpw/raporty-okresowe-i-prezentacje/",
        2024: "https://grupakety.com/relacje-inwestorskie/grupa-kety-na-gpw/raporty-okresowe-i-prezentacje/",
    },

    "LPP": {
        2022: "https://www.lpp.com/relacje-inwestorskie/materialy-inwestorskie/prezentacje-wynikowe/",
        2023: "https://www.lpp.com/relacje-inwestorskie/materialy-inwestorskie/prezentacje-wynikowe/",
        2024: "https://www.lpp.com/relacje-inwestorskie/materialy-inwestorskie/prezentacje-wynikowe/",
    },

    "Orange Polska": {
        2022: "https://www.orange-ir.pl/pl/centrum-wynikow/",
        2023: "https://www.orange-ir.pl/pl/centrum-wynikow/",
        2024: "https://www.orange-ir.pl/pl/centrum-wynikow/",
    },

    "PGE": {
        2022: "https://www.gkpge.pl/dla-inwestorow/akcje/dane-finansowe/raporty-okresowe-za-2022-rok",
        2023: "https://www.gkpge.pl/dla-inwestorow/akcje/dane-finansowe/raporty-okresowe-za-2023-rok",
        2024: "https://www.gkpge.pl/dla-inwestorow/akcje/dane-finansowe/raporty-okresowe-za-2024-rok",
    },
}

# 3. MAPOWANIE TARGET -> POPRZEDNI MATERIAŁ

STANDARD_PERIODS = [
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q1",
        "Rok dokumentu": 2022,
        "Okres dokumentu": "2022 FY/Q4",
        "Frazy": [
            "4 kwartał 2022",
            "q4 2022",
            "q4 2022 results",
            "2022 results",
            "wyniki za 2022",
            "wyniki 2022",
            "rok 2022"
        ]
    },

    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q2",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023 Q1",
        "Frazy": [
            "1 kwartał 2023",
            "i kwartał 2023",
            "q1 2023",
            "1q 2023",
            "1q2023"
        ]
    },

    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q3",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023 H1/Q2",
        "Frazy": [
            "1 półrocze 2023",
            "i półrocze 2023",
            "h1 2023",
            "2 kwartał 2023",
            "ii kwartał 2023",
            "q2 2023",
            "2q 2023"
        ]
    },

    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q4",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023 Q3",
        "Frazy": [
            "3 kwartał 2023",
            "iii kwartał 2023",
            "q3 2023",
            "3q 2023",
            "3q2023"
        ]
    },

    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q1",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023 FY/Q4",
        "Frazy": [
            "4 kwartał 2023",
            "q4 2023",
            "q4 2023 results",
            "2023 results",
            "wyniki za 2023",
            "wyniki 2023",
            "rok 2023"
        ]
    },

    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q2",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 Q1",
        "Frazy": [
            "1 kwartał 2024",
            "i kwartał 2024",
            "q1 2024",
            "1q 2024",
            "1q2024"
        ]
    },

    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q3",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 H1/Q2",
        "Frazy": [
            "1 półrocze 2024",
            "i półrocze 2024",
            "h1 2024",
            "2 kwartał 2024",
            "ii kwartał 2024",
            "q2 2024",
            "2q 2024"
        ]
    },

    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q4",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 Q3",
        "Frazy": [
            "3 kwartał 2024",
            "iii kwartał 2024",
            "q3 2024",
            "3q 2024",
            "3q2024"
        ]
    },
]

# 4. SPECJALNE FRAZY LPP

LPP_PERIODS = [
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q1",
        "Rok dokumentu": 2022,
        "Okres dokumentu": "2022/23 Q4",
        "Frazy": [
            "4q2022/23",
            "4q22/23",
            "4q2022-23"
        ]
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q2",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023/24 Q1",
        "Frazy": [
            "1q2023/24",
            "1q23/24"
        ]
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q3",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023/24 Q2",
        "Frazy": [
            "2q2023/24",
            "2q23/24"
        ]
    },
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q4",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023/24 Q3",
        "Frazy": [
            "3q2023/24",
            "3q23/24"
        ]
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q1",
        "Rok dokumentu": 2023,
        "Okres dokumentu": "2023 Q4",
        "Frazy": [
            "4q2023",
            "4q23"
        ]
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q2",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 Q1",
        "Frazy": [
            "1q2024",
            "1q24"
        ]
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q3",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 Q2",
        "Frazy": [
            "2q2024",
            "2q24"
        ]
    },
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q4",
        "Rok dokumentu": 2024,
        "Okres dokumentu": "2024 Q3",
        "Frazy": [
            "3q2024",
            "3q24"
        ]
    },
]


# 5. NORMALIZACJA

def normalize(text):

    if text is None:
        return ""

    text = str(text).lower().replace("\xa0", " ")

    # usuwamy polskie znaki tylko do wyszukiwania
    text = unicodedata.normalize("NFKD", text)
    text = "".join(
        c for c in text
        if not unicodedata.combining(c)
    )

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# 6. PUNKTACJA KANDYDATA

def score_candidate(text, phrases):

    text_n = normalize(text)

    score = 0

    for phrase in phrases:

        phrase_n = normalize(phrase)

        if phrase_n in text_n:
            score += 100

        # dodatkowe punkty za poszczególne tokeny
        for token in phrase_n.split():

            if len(token) >= 2 and token in text_n:
                score += 4

    positives = [
        "prezentac",
        "wynik",
        "results",
        "earnings",
        "financial",
        "komentarz"
    ]

    for word in positives:

        if word in text_n:
            score += 8

    negatives = [
        "strateg",
        "esg",
        "zrownowazon",
        "zaproszen",
        "walne",
        "dywidend",
        "szacunkowe",
        "estimate"
    ]

    for word in negatives:

        if word in text_n:
            score -= 15

    return score


# 7. LINKI NA STRONIE

def get_links(page_url):

    r = requests.get(
        page_url,
        headers=HEADERS,
        timeout=TIMEOUT
    )

    r.raise_for_status()

    soup = BeautifulSoup(
        r.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all("a", href=True):

        href = a.get("href", "")

        full_url = urljoin(
            page_url,
            href
        )

        label = a.get_text(
            " ",
            strip=True
        )

        parent_text = ""

        parent = a.find_parent(
            ["tr", "li", "article", "div"]
        )

        if parent is not None:
            parent_text = parent.get_text(
                " ",
                strip=True
            )

        combined = (
            label
            + " "
            + parent_text
            + " "
            + full_url
        )

        links.append({
            "url": full_url,
            "text": combined
        })

    return links


# 8. SZUKANIE PDF JEDEN POZIOM GŁĘBIEJ

def find_pdf_inside(page_url, phrases):

    try:

        links = get_links(page_url)

    except Exception:
        return None

    candidates = []

    for link in links:

        if ".pdf" not in link["url"].lower():
            continue

        score = score_candidate(
            link["text"],
            phrases
        )

        candidates.append(
            (score, link["url"])
        )

    if not candidates:
        return None

    candidates.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    return candidates[0][1]


# 9. ZNAJDOWANIE NAJLEPSZEGO DOKUMENTU

def resolve_document(page_url, phrases):

    links = get_links(page_url)

    scored = []

    for link in links:

        score = score_candidate(
            link["text"],
            phrases
        )

        if score > 0:

            scored.append(
                (
                    score,
                    link["url"],
                    link["text"]
                )
            )

    scored.sort(
        reverse=True,
        key=lambda x: x[0]
    )

    # testujemy najlepszych kandydatów
    for score, candidate_url, text in scored[:12]:

        # bezpośredni PDF
        if ".pdf" in candidate_url.lower():

            return {
                "pdf_url": candidate_url,
                "score": score,
                "candidate_text": text
            }

        # strona szczegółowa
        pdf_url = find_pdf_inside(
            candidate_url,
            phrases
        )

        if pdf_url:

            return {
                "pdf_url": pdf_url,
                "score": score,
                "candidate_text": text
            }

    return None


# 10. EKSTRAKCJA PDF

def extract_pdf_text(pdf_path):

    doc = pymupdf.open(pdf_path)

    pages = []

    for page in doc:

        text = page.get_text("text")

        text = re.sub(
            r"[ \t]+",
            " ",
            text
        )

        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )

        pages.append(
            text.strip()
        )

    doc.close()

    return "\n\n".join(pages)

# 10A. CZYSZCZENIE ZNAKÓW NIEDOZWOLONYCH W EXCELU

ILLEGAL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F]"
)

def clean_excel_value(value):
    """
    Usuwa z tekstu znaki sterujące niedozwolone w plikach XLSX.
    Nie zmienia treści merytorycznej dokumentu.
    """
    if isinstance(value, str):
        return ILLEGAL_CHARACTERS_RE.sub("", value)
    return value


def clean_dataframe_for_excel(dataframe):
    """
    Zwraca kopię DataFrame bez niedozwolonych znaków sterujących
    w komórkach tekstowych, gotową do zapisu przez openpyxl.
    """
    return dataframe.map(clean_excel_value)


# 11. BUDOWA 64 ZADAŃ

tasks = []

for company in SOURCES.keys():

    periods = (
        LPP_PERIODS
        if company == "LPP"
        else STANDARD_PERIODS
    )

    for p in periods:

        tasks.append({
            "Spółka": company,
            **p
        })


print("Liczba zadań:", len(tasks))


# 12. POBIERANIE

results = []

for i, task in enumerate(tasks, start=1):

    company = task["Spółka"]
    target_year = task["Rok prognozy"]
    target_quarter = task["Kwartał prognozy"]
    doc_year = task["Rok dokumentu"]
    source_period = task["Okres dokumentu"]
    phrases = task["Frazy"]

    archive_page = SOURCES[
        company
    ][
        doc_year
    ]

    print()
    print("=" * 78)

    print(
        f"{i}/{len(tasks)} | "
        f"{company} | "
        f"target {target_year} {target_quarter} | "
        f"source {source_period}"
    )

    print("Strona:")
    print(archive_page)

    try:

        resolved = resolve_document(
            archive_page,
            phrases
        )

        if resolved is None:

            raise Exception(
                "Nie udało się automatycznie znaleźć odpowiedniego PDF"
            )

        pdf_url = resolved["pdf_url"]

        print("PDF:")
        print(pdf_url)

        # bezpieczna nazwa spółki
        safe_company = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            company
        )

        file_base = (
            f"{safe_company}_"
            f"{target_year}_"
            f"{target_quarter}"
        )

        pdf_path = os.path.join(
            OUTPUT_FOLDER,
            file_base + ".pdf"
        )

        txt_path = os.path.join(
            OUTPUT_FOLDER,
            file_base + ".txt"
        )

        # pobieranie
        r = requests.get(
            pdf_url,
            headers=HEADERS,
            timeout=120
        )

        print(
            "Status PDF:",
            r.status_code
        )

        r.raise_for_status()

        with open(
            pdf_path,
            "wb"
        ) as f:
            f.write(r.content)

        # sprawdzamy czy to faktycznie PDF
        try:

            text = extract_pdf_text(
                pdf_path
            )

        except Exception as e:

            raise Exception(
                "Pobrany plik nie jest prawidłowym PDF: "
                + str(e)
            )

        with open(
            txt_path,
            "w",
            encoding="utf-8"
        ) as f:
            f.write(text)

        char_count = len(text)
        word_count = len(text.split())

        print(
            "Znaki:",
            char_count,
            "| Słowa:",
            word_count
        )

        # kontrola sensownej ilości tekstu
        if word_count < 200:

            status = "KONTROLA - bardzo krótki tekst"

        else:

            status = "OK"

        results.append({

            "Spółka": company,

            "Rok prognozy":
                target_year,

            "Kwartał prognozy":
                target_quarter,

            "Okres dokumentu":
                source_period,

            "Status":
                status,

            "Strona źródłowa":
                archive_page,

            "URL":
                pdf_url,

            "Score":
                resolved["score"],

            "Liczba znaków":
                char_count,

            "Liczba słów":
                word_count,

            "Plik PDF":
                pdf_path,

            "Plik TXT":
                txt_path,

            "Tekst":
                text
        })


    except Exception as e:

        print(
            "BŁĄD:",
            e
        )

        results.append({

            "Spółka": company,

            "Rok prognozy":
                target_year,

            "Kwartał prognozy":
                target_quarter,

            "Okres dokumentu":
                source_period,

            "Status":
                "BŁĄD: " + str(e),

            "Strona źródłowa":
                archive_page,

            "URL":
                "",

            "Score":
                np.nan if "np" in globals() else None,

            "Liczba znaków":
                0,

            "Liczba słów":
                0,

            "Plik PDF":
                "",

            "Plik TXT":
                "",

            "Tekst":
                ""
        })


    
    temp_df = pd.DataFrame(
        results
    )

    temp_df_clean = clean_dataframe_for_excel(
        temp_df
    )

    temp_df_clean.to_excel(
        OUTPUT_EXCEL,
        sheet_name="Materiały",
        index=False
    )

    time.sleep(0.8)



result_df = pd.DataFrame(
    results
)

print()
print("=" * 78)
print("PODSUMOWANIE")
print("=" * 78)

summary = (
    result_df
    .groupby(
        ["Spółka", "Status"]
    )
    .size()
    .reset_index(
        name="Liczba"
    )
)

display(summary)


print()
print("POZYCJE WYMAGAJĄCE KONTROLI:")

problems = result_df[
    result_df["Status"] != "OK"
][
    [
        "Spółka",
        "Rok prognozy",
        "Kwartał prognozy",
        "Okres dokumentu",
        "Status"
    ]
]

display(problems)


# 14. FINALNY ZAPIS

with pd.ExcelWriter(
    OUTPUT_EXCEL,
    engine="openpyxl"
) as writer:

    clean_dataframe_for_excel(
        result_df
    ).to_excel(
        writer,
        sheet_name="Materiały",
        index=False
    )

    clean_dataframe_for_excel(
        summary
    ).to_excel(
        writer,
        sheet_name="Podsumowanie",
        index=False
    )

    clean_dataframe_for_excel(
        problems
    ).to_excel(
        writer,
        sheet_name="Do kontroli",
        index=False
    )


print()
print("Gotowy plik:")
print(
    os.path.abspath(
        OUTPUT_EXCEL
    )
)

print()
print("Folder z PDF/TXT:")
print(
    os.path.abspath(
        OUTPUT_FOLDER
    )
)



In [ ]:
# Naprawa i walidacja materiałów tekstowych

import os
import re
import time
import requests
import pandas as pd
import pymupdf

from bs4 import BeautifulSoup
from urllib.parse import urljoin

# 1. PLIKI

INPUT_FILE = "LLM2_materialy_8_spolek.xlsx"

OUTPUT_FILE = "LLM2_materialy_8_spolek_FINAL.xlsx"

OUTPUT_FOLDER = "LLM2_materialy_8_spolek"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/125 Safari/537.36"
    )
}

# 2. WCZYTANIE DOTYCHCZASOWEGO PLIKU

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Materiały"
)

print("Liczba rekordów:", len(df))

# 3. USUWANIE NIEDOZWOLONYCH ZNAKÓW EXCELA

ILLEGAL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F]"
)

def clean_excel_value(value):

    if isinstance(value, str):

        return ILLEGAL_CHARACTERS_RE.sub(
            "",
            value
        )

    return value

# 4. EKSTRAKCJA TEKSTU Z PDF

def extract_pdf_text(pdf_path):

    doc = pymupdf.open(
        pdf_path
    )

    pages = []

    for page in doc:

        text = page.get_text(
            "text"
        )

        text = re.sub(
            r"[ \t]+",
            " ",
            text
        )

        text = re.sub(
            r"\n{3,}",
            "\n\n",
            text
        )

        pages.append(
            text.strip()
        )

    doc.close()

    return "\n\n".join(
        pages
    )

# 5. POBIERANIE JEDNEGO PDF

def download_document(
    company,
    year,
    quarter,
    pdf_url
):

    safe_company = re.sub(
        r"[^A-Za-z0-9_-]+",
        "_",
        company
    )

    file_base = (
        f"{safe_company}_{year}_{quarter}_FIX"
    )

    pdf_path = os.path.join(
        OUTPUT_FOLDER,
        file_base + ".pdf"
    )

    txt_path = os.path.join(
        OUTPUT_FOLDER,
        file_base + ".txt"
    )

    print("\nPobieram:")
    print(pdf_url)

    r = requests.get(
        pdf_url,
        headers=HEADERS,
        timeout=120
    )

    print(
        "Status:",
        r.status_code
    )

    r.raise_for_status()

    with open(
        pdf_path,
        "wb"
    ) as f:

        f.write(
            r.content
        )

    text = extract_pdf_text(
        pdf_path
    )

    with open(
        txt_path,
        "w",
        encoding="utf-8"
    ) as f:

        f.write(
            text
        )

    return {
        "URL": pdf_url,
        "Plik PDF": pdf_path,
        "Plik TXT": txt_path,
        "Liczba znaków": len(text),
        "Liczba słów": len(text.split()),
        "Tekst": text
    }

# 6. ORANGE POLSKA

orange_documents = [

    # target 2023 Q1 <- FY/Q4 2022
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2022 FY/Q4",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2023/02/RB-5-2023-wybrane-dane-za-4kw.-i-caly-2022-r.pdf"
        )
    },

    # target 2023 Q2 <- Q1 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2023 Q1",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2023/04/RB-7-2023-Wybrane-dane-1kw2023.pdf"
        )
    },

    # target 2023 Q3 <- H1 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2023 H1/Q2",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2023/07/sprawozdanie-finansowe-OPL-2Q-2023.pdf"
        )
    },

    # target 2023 Q4 <- Q3 2023
    {
        "Rok prognozy": 2023,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2023 Q3",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2023/10/RB-19-2023-wybrane-dane-3kw2023.pdf"
        )
    },

    # target 2024 Q1 <- FY/Q4 2023
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q1",
        "Okres dokumentu": "2023 FY/Q4",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2024/02/RB-4-2024-Wybrane-dane.pdf"
        )
    },

    # target 2024 Q2 <- Q1 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q2",
        "Okres dokumentu": "2024 Q1",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2024/04/RB-14-2024-Wybrane-dane.pdf"
        )
    },

    # target 2024 Q3 <- H1 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q3",
        "Okres dokumentu": "2024 H1/Q2",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2024/07/RB-17-2024-Wybrane-dane.pdf"
        )
    },

    # target 2024 Q4 <- Q3 2024
    {
        "Rok prognozy": 2024,
        "Kwartał prognozy": "Q4",
        "Okres dokumentu": "2024 Q3",
        "URL": (
            "https://www.orange-ir.pl/wp-content/uploads/"
            "2024/10/RB-18-2024-Wybrane-dane.pdf"
        )
    }
]

# 7. PODMIANA 8 REKORDÓW ORANGE

for i, item in enumerate(
    orange_documents,
    start=1
):

    year = item[
        "Rok prognozy"
    ]

    quarter = item[
        "Kwartał prognozy"
    ]

    print()
    print(
        "=" * 70
    )

    print(
        f"ORANGE {i}/8 | "
        f"{year} {quarter}"
    )

    try:

        result = download_document(
            "Orange Polska",
            year,
            quarter,
            item["URL"]
        )

        mask = (
            (df["Spółka"] == "Orange Polska")
            &
            (df["Rok prognozy"] == year)
            &
            (df["Kwartał prognozy"] == quarter)
        )

        if mask.sum() != 1:

            print(
                "UWAGA: znaleziono rekordów:",
                mask.sum()
            )

            continue

        idx = df[
            mask
        ].index[0]

        df.loc[
            idx,
            "Okres dokumentu"
        ] = item[
            "Okres dokumentu"
        ]

        df.loc[
            idx,
            "Status"
        ] = "OK"

        df.loc[
            idx,
            "URL"
        ] = result[
            "URL"
        ]

        df.loc[
            idx,
            "Plik PDF"
        ] = result[
            "Plik PDF"
        ]

        df.loc[
            idx,
            "Plik TXT"
        ] = result[
            "Plik TXT"
        ]

        df.loc[
            idx,
            "Liczba znaków"
        ] = result[
            "Liczba znaków"
        ]

        df.loc[
            idx,
            "Liczba słów"
        ] = result[
            "Liczba słów"
        ]

        df.loc[
            idx,
            "Tekst"
        ] = result[
            "Tekst"
        ]

        print(
            "Słowa:",
            result["Liczba słów"]
        )

    except Exception as e:

        print(
            "BŁĄD ORANGE:",
            e
        )

    time.sleep(
        0.5
    )


# 8. CD PROJEKT FY2022

CDP_PAGE = (
    "https://www.cdprojekt.com/pl/inwestorzy/"
    "prezentacje/prezentacja-grupy-cd-projekt-wyniki-2022-en/"
)

print()
print(
    "=" * 70
)
print(
    "CD PROJEKT FY2022"
)
print(
    "=" * 70
)

try:

    r = requests.get(
        CDP_PAGE,
        headers=HEADERS,
        timeout=60
    )

    print(
        "Status strony:",
        r.status_code
    )

    r.raise_for_status()

    soup = BeautifulSoup(
        r.text,
        "html.parser"
    )

    pdf_candidates = []

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get(
            "href",
            ""
        )

        full_url = urljoin(
            CDP_PAGE,
            href
        )

        label = a.get_text(
            " ",
            strip=True
        )

        if ".pdf" in full_url.lower():

            combined = (
                label
                + " "
                + full_url
            ).lower()

            score = 0

            if "presentation" in combined:
                score += 10

            if "prezentac" in combined:
                score += 10

            if "2022" in combined:
                score += 10

            if "fy" in combined:
                score += 5

            pdf_candidates.append(
                (
                    score,
                    full_url
                )
            )


    if not pdf_candidates:

        raise Exception(
            "Nie znaleziono PDF FY2022"
        )

    pdf_candidates.sort(
        key=lambda x: x[0],
        reverse=True
    )

    cdp_pdf_url = (
        pdf_candidates[0][1]
    )

    print(
        "Wybrany PDF:"
    )
    print(
        cdp_pdf_url
    )

    result = download_document(
        "CD Projekt",
        2023,
        "Q1",
        cdp_pdf_url
    )

    mask = (
        (df["Spółka"] == "CD Projekt")
        &
        (df["Rok prognozy"] == 2023)
        &
        (df["Kwartał prognozy"] == "Q1")
    )

    idx = df[
        mask
    ].index[0]

    df.loc[
        idx,
        "Status"
    ] = "OK"

    df.loc[
        idx,
        "URL"
    ] = result[
        "URL"
    ]

    df.loc[
        idx,
        "Plik PDF"
    ] = result[
        "Plik PDF"
    ]

    df.loc[
        idx,
        "Plik TXT"
    ] = result[
        "Plik TXT"
    ]

    df.loc[
        idx,
        "Liczba znaków"
    ] = result[
        "Liczba znaków"
    ]

    df.loc[
        idx,
        "Liczba słów"
    ] = result[
        "Liczba słów"
    ]

    df.loc[
        idx,
        "Tekst"
    ] = result[
        "Tekst"
    ]

    print(
        "CD Projekt FY2022 - słowa:",
        result["Liczba słów"]
    )


except Exception as e:

    print(
        "BŁĄD CD PROJEKT:",
        e
    )

# 9. KONTROLA 64 REKORDÓW

print()
print(
    "=" * 70
)
print(
    "KONTROLA KOŃCOWA"
)
print(
    "=" * 70
)

summary = (
    df
    .groupby(
        [
            "Spółka",
            "Status"
        ]
    )
    .size()
    .reset_index(
        name="Liczba"
    )
)

display(
    summary
)

# 10. POZYCJE NADAL NIEPOPRAWNE

problems = df[
    df["Status"] != "OK"
].copy()

print(
    "\nLiczba rekordów wymagających dalszej kontroli:",
    len(problems)
)

display(
    problems[
        [
            "Spółka",
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "Status",
            "Liczba słów"
        ]
    ]
)

# 11. KONTROLA DUPLIKATÓW URL

duplicate_urls = df[
    df["URL"].notna()
    &
    (df["URL"].astype(str) != "")
    &
    df.duplicated(
        subset=[
            "Spółka",
            "URL"
        ],
        keep=False
    )
].sort_values(
    [
        "Spółka",
        "URL",
        "Rok prognozy",
        "Kwartał prognozy"
    ]
)

print(
    "\nPowtarzające się URL w obrębie jednej spółki:",
    len(duplicate_urls)
)

if len(
    duplicate_urls
) > 0:

    display(
        duplicate_urls[
            [
                "Spółka",
                "Rok prognozy",
                "Kwartał prognozy",
                "Okres dokumentu",
                "URL"
            ]
        ]
    )


# 12. CZYSZCZENIE ZNAKÓW PRZED ZAPISEM

df_clean = df.apply(
    lambda col:
        col.map(
            clean_excel_value
        )
)

summary_clean = summary.apply(
    lambda col:
        col.map(
            clean_excel_value
        )
)

problems_clean = problems.apply(
    lambda col:
        col.map(
            clean_excel_value
        )
)

duplicate_urls_clean = (
    duplicate_urls.apply(
        lambda col:
            col.map(
                clean_excel_value
            )
    )
)


# 13. FINALNY ZAPIS

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    df_clean.to_excel(
        writer,
        sheet_name="Materiały",
        index=False
    )

    summary_clean.to_excel(
        writer,
        sheet_name="Podsumowanie",
        index=False
    )

    problems_clean.to_excel(
        writer,
        sheet_name="Do kontroli",
        index=False
    )

    duplicate_urls_clean.to_excel(
        writer,
        sheet_name="Duplikaty URL",
        index=False
    )


print()
print(
    "=" * 70
)
print(
    "GOTOWE"
)
print(
    "=" * 70
)

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)



In [ ]:
# Dodatkowa walidacja i naprawa powtarzających się materiałów PDF

import os
import re
import time
import unicodedata
from io import BytesIO
from urllib.parse import urljoin

import requests
import pandas as pd
from bs4 import BeautifulSoup
import pymupdf


# 1. USTAWIENIA

INPUT_FILE = "LLM2_materialy_8_spolek_FINAL.xlsx"
OUTPUT_FILE = "LLM2_materialy_8_spolek_FINAL_v2.xlsx"

OUTPUT_FOLDER = "LLM2_materialy_REPAIR"

os.makedirs(
    OUTPUT_FOLDER,
    exist_ok=True
)

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/125 Safari/537.36"
    )
}

TIMEOUT = 90

# 2. WCZYTANIE

df = pd.read_excel(
    INPUT_FILE,
    sheet_name="Materiały"
)

print("Liczba rekordów:", len(df))

# 3. NORMALIZACJA TEKSTU

def normalize(text):

    if text is None:
        return ""

    text = str(text).lower()

    text = unicodedata.normalize(
        "NFKD",
        text
    )

    text = "".join(
        char
        for char in text
        if not unicodedata.combining(char)
    )

    text = text.replace("\xa0", " ")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


# 4. FRAZY OCZEKIWANE DLA DANEGO OKRESU

def get_period_phrases(period):

    p = normalize(period)

    year_match = re.search(
        r"20\d{2}",
        p
    )

    year = (
        year_match.group()
        if year_match
        else ""
    )

    if "q1" in p:

        return [
            f"q1 {year}",
            f"q1{year}",
            f"1 kwartal {year}",
            f"i kwartal {year}",
            f"pierwszy kwartal {year}"
        ]

    if (
        "h1" in p
        or "q2" in p
    ):

        return [
            f"h1 {year}",
            f"h1{year}",
            f"1 polrocze {year}",
            f"i polrocze {year}",
            f"pierwsze polrocze {year}",
            f"q2 {year}",
            f"q2{year}",
            f"2 kwartal {year}",
            f"ii kwartal {year}"
        ]

    if "q3" in p:

        return [
            f"q3 {year}",
            f"q3{year}",
            f"3 kwartal {year}",
            f"iii kwartal {year}",
            f"trzeci kwartal {year}",
            f"trzy kwartaly {year}"
        ]

    # FY / Q4
    return [
        f"q4 {year}",
        f"q4{year}",
        f"4 kwartal {year}",
        f"iv kwartal {year}",
        f"rok {year}",
        f"za rok {year}",
        f"wyniki {year}",
        f"results {year}"
    ]


# 5. POBRANIE LINKÓW ZE STRONY

def get_links(page_url):

    response = requests.get(
        page_url,
        headers=HEADERS,
        timeout=TIMEOUT
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    links = []

    for a in soup.find_all(
        "a",
        href=True
    ):

        href = a.get(
            "href",
            ""
        )

        full_url = urljoin(
            page_url,
            href
        )

        label = a.get_text(
            " ",
            strip=True
        )

        parent_text = ""

        parent = a.find_parent(
            [
                "tr",
                "li",
                "article",
                "section",
                "div"
            ]
        )

        if parent is not None:

            parent_text = parent.get_text(
                " ",
                strip=True
            )

        context = (
            label
            + " "
            + parent_text
        )

        links.append({
            "url": full_url,
            "context": context
        })

    return links


# 6. ZBIERANIE KANDYDATÓW PDF

def collect_pdf_candidates(page_url):

    candidates = {}

    try:

        links = get_links(
            page_url
        )

    except Exception:

        return []

    # bezpośrednie PDF
    for link in links:

        if ".pdf" in link["url"].lower():

            candidates[
                link["url"]
            ] = link["context"]


    # strony szczegółowe
    non_pdf_links = [
        link
        for link in links
        if (
            ".pdf" not in link["url"].lower()
            and link["url"].startswith("http")
        )
    ]

    # ograniczenie liczby zapytań
    for link in non_pdf_links[:40]:

        try:

            inner_links = get_links(
                link["url"]
            )

            for inner in inner_links:

                if ".pdf" in inner["url"].lower():

                    context = (
                        link["context"]
                        + " "
                        + inner["context"]
                    )

                    candidates[
                        inner["url"]
                    ] = context

        except Exception:

            continue

    return [
        {
            "url": url,
            "context": context
        }
        for url, context
        in candidates.items()
    ]


# 7. EKSTRAKCJA PIERWSZYCH STRON PDF

def extract_preview(pdf_bytes, max_pages=5):

    doc = pymupdf.open(
        stream=pdf_bytes,
        filetype="pdf"
    )

    texts = []

    page_count = min(
        max_pages,
        len(doc)
    )

    for i in range(
        page_count
    ):

        texts.append(
            doc[i].get_text(
                "text"
            )
        )

    doc.close()

    return "\n".join(
        texts
    )


# 8. PEŁNA EKSTRAKCJA

def extract_full_text(pdf_bytes):

    doc = pymupdf.open(
        stream=pdf_bytes,
        filetype="pdf"
    )

    texts = []

    for page in doc:

        texts.append(
            page.get_text(
                "text"
            )
        )

    doc.close()

    text = "\n".join(
        texts
    )

    text = re.sub(
        r"[ \t]+",
        " ",
        text
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()


# 9. OCENA PDF-U

def score_pdf(
    preview,
    context,
    expected_period
):

    preview_n = normalize(
        preview
    )

    context_n = normalize(
        context
    )

    phrases = get_period_phrases(
        expected_period
    )

    score = 0

    # okres na pierwszych stronach jest najważniejszy
    for phrase in phrases:

        phrase_n = normalize(
            phrase
        )

        if phrase_n in preview_n:

            score += 100

        if phrase_n in context_n:

            score += 50


    # preferujemy prezentacje wynikowe
    positive_words = [
        "prezentac",
        "presentation",
        "wyniki",
        "results",
        "financial results",
        "wynikow finansowych"
    ]

    for word in positive_words:

        word_n = normalize(
            word
        )

        if word_n in preview_n:
            score += 15

        if word_n in context_n:
            score += 10


    # nie chcemy np. ESG / strategii / raportów bieżących
    negative_words = [
        "esg",
        "strategia",
        "sustainability",
        "dywidenda",
        "walne zgromadzenie",
        "remuneration",
        "wynagrodzen"
    ]

    for word in negative_words:

        if normalize(word) in preview_n:
            score -= 30


    return score


# 10. WYBÓR POPRAWNEGO PDF

def find_best_pdf(
    page_url,
    expected_period
):

    candidates = collect_pdf_candidates(
        page_url
    )

    print(
        "  Kandydatów PDF:",
        len(candidates)
    )

    evaluated = []

    for candidate in candidates:

        try:

            response = requests.get(
                candidate["url"],
                headers=HEADERS,
                timeout=120
            )

            if response.status_code != 200:
                continue

            # szybkie sprawdzenie PDF
            if not response.content.startswith(
                b"%PDF"
            ):
                continue

            preview = extract_preview(
                response.content,
                max_pages=5
            )

            score = score_pdf(
                preview,
                candidate["context"],
                expected_period
            )

            evaluated.append({
                "score": score,
                "url": candidate["url"],
                "bytes": response.content,
                "preview": preview
            })

        except Exception:

            continue


    if not evaluated:

        return None


    evaluated.sort(
        key=lambda x: x["score"],
        reverse=True
    )


    # pokaż top 3 dla kontroli
    print("  TOP kandydaci:")

    for item in evaluated[:3]:

        print(
            "   ",
            item["score"],
            "|",
            item["url"][:120]
        )


    best = evaluated[0]

    # wymagamy sensownego dopasowania
    if best["score"] < 80:

        return None

    return best


# 11. IDENTYFIKACJA DUPLIKATÓW

duplicate_mask = (
    df["URL"].notna()
    &
    (df["URL"].astype(str) != "")
    &
    df.duplicated(
        subset=[
            "Spółka",
            "URL"
        ],
        keep=False
    )
)

duplicates = df[
    duplicate_mask
].copy()


print(
    "\nLiczba rekordów do naprawy:",
    len(duplicates)
)

print(
    "Spółki:",
    duplicates["Spółka"].unique()
)


# 12. NAPRAWA

for idx in duplicates.index:

    company = df.loc[
        idx,
        "Spółka"
    ]

    year = df.loc[
        idx,
        "Rok prognozy"
    ]

    quarter = df.loc[
        idx,
        "Kwartał prognozy"
    ]

    period = df.loc[
        idx,
        "Okres dokumentu"
    ]

    page_url = df.loc[
        idx,
        "Strona źródłowa"
    ]


    print()
    print("=" * 75)

    print(
        company,
        "| target",
        year,
        quarter,
        "| source",
        period
    )


    try:

        best = find_best_pdf(
            page_url,
            period
        )

        if best is None:

            print(
                "  NIE UDAŁO SIĘ JEDNOZNACZNIE DOPASOWAĆ."
            )

            df.loc[
                idx,
                "Status"
            ] = "DO RĘCZNEJ KONTROLI"

            continue


        pdf_bytes = best[
            "bytes"
        ]

        pdf_url = best[
            "url"
        ]

        full_text = extract_full_text(
            pdf_bytes
        )


        safe_company = re.sub(
            r"[^A-Za-z0-9_-]+",
            "_",
            company
        )

        file_base = (
            f"{safe_company}_"
            f"{year}_"
            f"{quarter}_REPAIR"
        )

        pdf_path = os.path.join(
            OUTPUT_FOLDER,
            file_base + ".pdf"
        )

        txt_path = os.path.join(
            OUTPUT_FOLDER,
            file_base + ".txt"
        )


        with open(
            pdf_path,
            "wb"
        ) as f:

            f.write(
                pdf_bytes
            )


        with open(
            txt_path,
            "w",
            encoding="utf-8"
        ) as f:

            f.write(
                full_text
            )


        df.loc[
            idx,
            "URL"
        ] = pdf_url

        df.loc[
            idx,
            "Plik PDF"
        ] = pdf_path

        df.loc[
            idx,
            "Plik TXT"
        ] = txt_path

        df.loc[
            idx,
            "Liczba znaków"
        ] = len(
            full_text
        )

        df.loc[
            idx,
            "Liczba słów"
        ] = len(
            full_text.split()
        )

        df.loc[
            idx,
            "Tekst"
        ] = full_text

        df.loc[
            idx,
            "Status"
        ] = "OK"


        print(
            "  WYBRANO:",
            pdf_url
        )

        print(
            "  słów:",
            len(
                full_text.split()
            )
        )


    except Exception as e:

        print(
            "  BŁĄD:",
            e
        )

        df.loc[
            idx,
            "Status"
        ] = (
            "DO RĘCZNEJ KONTROLI: "
            + str(e)
        )


    time.sleep(
        0.5
    )


# 13. PONOWNA KONTROLA DUPLIKATÓW

duplicate_after = df[
    df["URL"].notna()
    &
    (df["URL"].astype(str) != "")
    &
    df.duplicated(
        subset=[
            "Spółka",
            "URL"
        ],
        keep=False
    )
].copy()


bad_status = df[
    df["Status"] != "OK"
].copy()


print()
print("=" * 75)
print("KONTROLA PO NAPRAWIE")
print("=" * 75)

print(
    "Powtarzające się URL:",
    len(
        duplicate_after
    )
)

print(
    "Rekordy Status != OK:",
    len(
        bad_status
    )
)


if len(
    duplicate_after
) > 0:

    display(
        duplicate_after[
            [
                "Spółka",
                "Rok prognozy",
                "Kwartał prognozy",
                "Okres dokumentu",
                "URL"
            ]
        ]
    )


if len(
    bad_status
) > 0:

    display(
        bad_status[
            [
                "Spółka",
                "Rok prognozy",
                "Kwartał prognozy",
                "Okres dokumentu",
                "Status"
            ]
        ]
    )


# 14. CZYSZCZENIE ZNAKÓW EXCEL

ILLEGAL_CHARACTERS_RE = re.compile(
    r"[\x00-\x08\x0B\x0C\x0E-\x1F]"
)


def clean_excel(value):

    if isinstance(
        value,
        str
    ):

        return ILLEGAL_CHARACTERS_RE.sub(
            "",
            value
        )

    return value


df_clean = df.apply(
    lambda col:
        col.map(
            clean_excel
        )
)


# 15. ZAPIS

with pd.ExcelWriter(
    OUTPUT_FILE,
    engine="openpyxl"
) as writer:

    df_clean.to_excel(
        writer,
        sheet_name="Materiały",
        index=False
    )

    duplicate_after.to_excel(
        writer,
        sheet_name="Duplikaty po naprawie",
        index=False
    )

    bad_status.to_excel(
        writer,
        sheet_name="Do kontroli",
        index=False
    )


print()
print("GOTOWE:")
print(
    os.path.abspath(
        OUTPUT_FILE
    )
)


In [ ]:
# FINALNY EKSPERYMENT LLM-2
# 8 POZOSTAŁYCH SPÓŁEK = 64 PROGNOZY

import os
import re
import time
import requests
import pandas as pd
import numpy as np

# 1. USTAWIENIA

PROMPTS_FILE = "dane_wejsciowe_LLM_liczby_v3.xlsx"

TEXTS_FILE = "LLM2_materialy_8_spolek_FINAL_v2.xlsx"

OUTPUT_FILE = "LLM2_wyniki_8_spolek.xlsx"

MODEL = "gpt-5.6-luna"

URL = "https://api.openai.com/v1/responses"

HEADERS = {
    "Authorization": f"Bearer {os.environ['OPENAI_API_KEY']}",
    "Content-Type": "application/json"
}


# 2. WCZYTANIE PROMPTÓW LICZBOWYCH

prompts = pd.read_excel(
    PROMPTS_FILE,
    sheet_name="Prompty LLM"
)


# ORLEN i KGHM mamy już wykonane osobno
companies_to_run = [
    "Asseco Poland",
    "Budimex",
    "CD Projekt",
    "Cyfrowy Polsat",
    "Grupa Kęty",
    "LPP",
    "Orange Polska",
    "PGE"
]

prompts_8 = prompts[
    prompts["Spółka rzeczywista"].isin(
        companies_to_run
    )
].copy()


print(
    "Liczba promptów liczbowych:",
    len(prompts_8)
)

# 3. WCZYTANIE MATERIAŁÓW TEKSTOWYCH

texts = pd.read_excel(
    TEXTS_FILE,
    sheet_name="Materiały"
)


texts_8 = texts[
    texts["Spółka"].isin(
        companies_to_run
    )
].copy()


print(
    "Liczba materiałów tekstowych:",
    len(texts_8)
)

# 4. KONTROLA STATUSÓW

bad_status = texts_8[
    texts_8["Status"] != "OK"
]

print(
    "Materiały ze statusem innym niż OK:",
    len(bad_status)
)

if len(bad_status) > 0:

    display(
        bad_status[
            [
                "Spółka",
                "Rok prognozy",
                "Kwartał prognozy",
                "Status"
            ]
        ]
    )

    raise ValueError(
        "Nie wszystkie materiały mają Status = OK."
    )

# 5. KONTROLA DUPLIKATÓW URL

duplicate_urls = texts_8[
    texts_8["URL"].notna()
    &
    (texts_8["URL"].astype(str) != "")
    &
    texts_8.duplicated(
        subset=[
            "Spółka",
            "URL"
        ],
        keep=False
    )
]


print(
    "Powtarzające się URL w obrębie spółki:",
    len(duplicate_urls)
)


if len(duplicate_urls) > 0:

    display(
        duplicate_urls[
            [
                "Spółka",
                "Rok prognozy",
                "Kwartał prognozy",
                "Okres dokumentu",
                "URL"
            ]
        ]
    )

    raise ValueError(
        "Wykryto powtarzające się URL. "
        "Sprawdź materiały przed uruchomieniem API."
    )

# 6. POŁĄCZENIE PROMPTÓW Z TEKSTAMI
merged = pd.merge(

    prompts_8,

    texts_8[
        [
            "Spółka",
            "Rok prognozy",
            "Kwartał prognozy",
            "Okres dokumentu",
            "URL",
            "Liczba znaków",
            "Liczba słów",
            "Tekst"
        ]
    ],

    left_on=[
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy"
    ],

    right_on=[
        "Spółka",
        "Rok prognozy",
        "Kwartał prognozy"
    ],

    how="left"
)


print(
    "Liczba rekordów po połączeniu:",
    len(merged)
)

print(
    "Brakujące teksty:",
    merged["Tekst"].isna().sum()
)


if len(merged) != 64:

    raise ValueError(
        f"Oczekiwano 64 rekordów, otrzymano {len(merged)}."
    )


if merged["Tekst"].isna().sum() != 0:

    raise ValueError(
        "Występują brakujące teksty."
    )

# 7. CZYSZCZENIE TEKSTU

def clean_text(text):

    if pd.isna(text):
        return ""

    lines = [
        line.strip()
        for line in str(text).splitlines()
        if line.strip()
    ]

    return "\n".join(lines)


merged["Tekst oczyszczony"] = (
    merged["Tekst"]
    .apply(clean_text)
)


# 8. BUDOWA PROMPTU LLM-2

def build_llm2_prompt(row):

    numeric_prompt = row["Prompt"]

    report_text = row[
        "Tekst oczyszczony"
    ]

    source_period = row[
        "Okres dokumentu"
    ]

    new_prompt = f"""
{numeric_prompt}

DODATKOWY KONTEKST TEKSTOWY:

Poniżej przedstawiono treść materiału wynikowego dotyczącego
poprzedniego okresu sprawozdawczego ({source_period}).

Materiał ten był dostępny w momencie wykonywania prognozy.

Wykorzystaj informacje tekstowe jedynie jako dodatkowy kontekst
do danych liczbowych. Zwróć szczególną uwagę na informacje dotyczące:

- zmian przychodów,
- rentowności i marż,
- kosztów działalności,
- sytuacji poszczególnych segmentów,
- czynników jednorazowych,
- inwestycji,
- warunków rynkowych,
- ryzyk,
- oczekiwań i perspektyw kolejnych okresów.

Nie zakładaj, że każda informacja zawarta w tekście musi mieć
bezpośredni wpływ na kolejny kwartał.

TREŚĆ MATERIAŁU:

--- POCZĄTEK TEKSTU ---

{report_text}

--- KONIEC TEKSTU ---

Na podstawie danych liczbowych oraz przedstawionego kontekstu tekstowego
oszacuj wyniki kolejnego kwartału.

Zwróć WYŁĄCZNIE:

PRZYCHODY: [pełna liczba całkowita w tys. PLN]
EBIT: [pełna liczba całkowita w tys. PLN]
ZYSK_NETTO: [pełna liczba całkowita w tys. PLN]

Nie dodawaj komentarza, uzasadnienia ani dodatkowego tekstu.
""".strip()

    return new_prompt


merged["Prompt LLM-2"] = merged.apply(
    build_llm2_prompt,
    axis=1
)

# 9. KOLUMNY NA WYNIKI

merged["LLM2 Przychody"] = np.nan
merged["LLM2 EBIT"] = np.nan
merged["LLM2 Zysk netto"] = np.nan

merged["Odpowiedź LLM2"] = ""


# 10. FUNKCJA WYCIĄGAJĄCA TEKST API

def extract_output_text(result):

    text = ""

    for item in result.get(
        "output",
        []
    ):

        if item.get(
            "type"
        ) == "message":

            for content in item.get(
                "content",
                []
            ):

                if content.get(
                    "type"
                ) == "output_text":

                    text += content.get(
                        "text",
                        ""
                    )

    return text.strip()

# 11. PARSER LICZBY

def parse_number(text):

    if text is None:
        return np.nan

    text = (
        str(text)
        .strip()
        .replace("\xa0", "")
        .replace(" ", "")
        .replace(",", "")
    )

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        text
    )

    if not match:
        return np.nan

    try:
        return float(
            match.group()
        )

    except:
        return np.nan

# 12. PARSER ODPOWIEDZI

def parse_response(text):

    revenue_match = re.search(
        r"PRZYCHODY\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    ebit_match = re.search(
        r"EBIT\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    net_match = re.search(
        r"ZYSK_NETTO\s*:\s*([^\n\r]+)",
        text,
        re.IGNORECASE
    )

    revenue = (
        parse_number(
            revenue_match.group(1)
        )
        if revenue_match
        else np.nan
    )

    ebit = (
        parse_number(
            ebit_match.group(1)
        )
        if ebit_match
        else np.nan
    )

    net_income = (
        parse_number(
            net_match.group(1)
        )
        if net_match
        else np.nan
    )

    return (
        revenue,
        ebit,
        net_income
    )

# 13. JEŻELI PLIK WYNIKOWY JUŻ ISTNIEJE,
#     MOŻEMY KONTYNUOWAĆ OD MIEJSCA PRZERWANIA

if os.path.exists(
    OUTPUT_FILE
):

    print(
        "\nZnaleziono wcześniejszy plik wynikowy."
    )

    old = pd.read_excel(
        OUTPUT_FILE,
        sheet_name="LLM2 wyniki"
    )

    if len(old) == len(merged):

        for col in [
            "LLM2 Przychody",
            "LLM2 EBIT",
            "LLM2 Zysk netto",
            "Odpowiedź LLM2"
        ]:

            if col in old.columns:

                merged[col] = old[col]

        print(
            "Wczytano dotychczasowe wyniki."
        )

# 14. WYKONANIE 64 PROGNOZ

for i in range(
    len(merged)
):

    # jeśli wszystkie 3 wartości już są,
    # pomijamy wiersz

    if (
        pd.notna(
            merged.loc[
                i,
                "LLM2 Przychody"
            ]
        )
        and
        pd.notna(
            merged.loc[
                i,
                "LLM2 EBIT"
            ]
        )
        and
        pd.notna(
            merged.loc[
                i,
                "LLM2 Zysk netto"
            ]
        )
    ):

        print(
            f"{i+1}/{len(merged)} "
            "- już wykonane"
        )

        continue


    company = merged.loc[
        i,
        "Spółka rzeczywista"
    ]

    year = merged.loc[
        i,
        "Rok prognozy"
    ]

    quarter = merged.loc[
        i,
        "Kwartał prognozy"
    ]

    prompt = merged.loc[
        i,
        "Prompt LLM-2"
    ]


    print()
    print(
        f"{i+1}/{len(merged)} | "
        f"{company} | "
        f"{year} {quarter}"
    )


    payload = {
        "model": MODEL,
        "input": prompt
    }


    try:

        response = requests.post(
            URL,
            headers=HEADERS,
            json=payload,
            timeout=240
        )

        print(
            "Status:",
            response.status_code
        )


        if response.status_code != 200:

            error_text = (
                response.text[:1000]
            )

            print(
                "BŁĄD API:",
                error_text
            )

            merged.loc[
                i,
                "Odpowiedź LLM2"
            ] = (
                "API_ERROR: "
                + error_text
            )


        else:

            result = response.json()

            raw_text = (
                extract_output_text(
                    result
                )
            )

            (
                revenue,
                ebit,
                net_income

            ) = parse_response(
                raw_text
            )


            merged.loc[
                i,
                "LLM2 Przychody"
            ] = revenue

            merged.loc[
                i,
                "LLM2 EBIT"
            ] = ebit

            merged.loc[
                i,
                "LLM2 Zysk netto"
            ] = net_income

            merged.loc[
                i,
                "Odpowiedź LLM2"
            ] = raw_text


            print(
                "Przychody:",
                revenue,
                "| EBIT:",
                ebit,
                "| Zysk netto:",
                net_income
            )


    except Exception as e:

        print(
            "WYJĄTEK:",
            e
        )

        merged.loc[
            i,
            "Odpowiedź LLM2"
        ] = (
            "EXCEPTION: "
            + str(e)
        )

    # ZAPIS PO KAŻDYM WYWOŁANIU

    merged.to_excel(
        OUTPUT_FILE,
        sheet_name="LLM2 wyniki",
        index=False
    )

    time.sleep(
        0.5
    )
# 15. KONTROLA KOŃCOWA
print()
print("=" * 70)
print("KONTROLA KOŃCOWA LLM-2")
print("=" * 70)

print(
    "Liczba obserwacji:",
    len(merged)
)

print()

print(
    "Braki prognoz:"
)

print(
    merged[
        [
            "LLM2 Przychody",
            "LLM2 EBIT",
            "LLM2 Zysk netto"
        ]
    ]
    .isna()
    .sum()
)


errors = (
    merged[
        "Odpowiedź LLM2"
    ]
    .astype(str)
    .str.contains(
        "API_ERROR|EXCEPTION",
        regex=True
    )
    .sum()
)


print(
    "\nBłędy API / wyjątki:",
    errors
)


print()
print(
    "Wyniki według spółek:"
)

print(
    merged
    .groupby(
        "Spółka rzeczywista"
    )
    .size()
)


print()
print("=" * 70)
print("GOTOWE")
print("=" * 70)

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)

In [ ]:
# Finalna analiza wszystkich pięciu modeli
# Wynik:
# FINALNE_wyniki_badania.xlsx

import os
import numpy as np
import pandas as pd


# 1. PLIKI

ML_FILE = "wyniki_modeli_ML_rozszerzone.xlsx"

LLM1_FILE = "wyniki_LLM_liczby_v3.xlsx"

ORLEN_FILE = "ORLEN_wyniki_LLM2.xlsx"

KGHM_FILE = "KGHM_wyniki_LLM2.xlsx"

LLM2_8_FILE = "LLM2_wyniki_8_spolek.xlsx"

OUTPUT_FILE = "FINALNE_wyniki_badania.xlsx"

# 2. FUNKCJE POMOCNICZE

def mae(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.mean(
        np.abs(
            y_true - y_pred
        )
    )


def rmse(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )


def smape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denominator = (
        np.abs(y_true)
        +
        np.abs(y_pred)
    )

    mask = (
        denominator != 0
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            2
            * np.abs(
                y_pred[mask]
                - y_true[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def wape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denominator = np.sum(
        np.abs(y_true)
    )

    if denominator == 0:
        return np.nan

    return (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        /
        denominator
        * 100
    )


# 3. FUNKCJA DO NORMALIZACJI NAZWY ZMIENNEJ

def normalize_variable(value):

    text = (
        str(value)
        .lower()
        .replace("_", " ")
        .strip()
    )

    if "przych" in text:
        return "Przychody"

    if "ebit" in text:
        return "EBIT"

    if "zysk" in text:
        return "Zysk netto"

    return str(value)


# 4. FUNKCJA PRZYGOTOWANIA LLM-2

def prepare_llm2(
    df,
    forced_company=None
):

    result = df.copy()

    # SPÓŁKA

    if forced_company is not None:

        result[
            "Spółka rzeczywista"
        ] = forced_company

    elif (
        "Spółka rzeczywista"
        not in result.columns
    ):

        if "Spółka" in result.columns:

            result[
                "Spółka rzeczywista"
            ] = result[
                "Spółka"
            ]

        else:

            raise ValueError(
                "Brak kolumny ze spółką."
            )


    # wymagane kolumny
  
    required = [

        "Spółka rzeczywista",

        "Rok prognozy",

        "Kwartał prognozy",

        "Przychody rzeczywiste",

        "EBIT rzeczywisty",

        "Zysk netto rzeczywisty",

        "LLM2 Przychody",

        "LLM2 EBIT",

        "LLM2 Zysk netto"

    ]


    missing = [

        col

        for col in required

        if col not in result.columns

    ]


    if missing:

        raise ValueError(
            f"Brakujące kolumny LLM-2: {missing}"
        )


    result = result[
        required
    ].copy()


    # numeryczne
    
    numeric_cols = [

        "Rok prognozy",

        "Przychody rzeczywiste",

        "EBIT rzeczywisty",

        "Zysk netto rzeczywisty",

        "LLM2 Przychody",

        "LLM2 EBIT",

        "LLM2 Zysk netto"

    ]


    for col in numeric_cols:

        result[col] = pd.to_numeric(
            result[col],
            errors="coerce"
        )


    result[
        "Kwartał prognozy"
    ] = (
        result[
            "Kwartał prognozy"
        ]
        .astype(str)
        .str.strip()
    )


    return result


# 5. WCZYTANIE ORLEN LLM-2

orlen = pd.read_excel(
    ORLEN_FILE,
    sheet_name="ORLEN LLM2"
)


orlen = prepare_llm2(
    orlen,
    forced_company="ORLEN"
)


print(
    "ORLEN:",
    len(orlen)
)


# 6. WCZYTANIE KGHM LLM-2

kghm = pd.read_excel(
    KGHM_FILE,
    sheet_name="KGHM LLM2"
)


kghm = prepare_llm2(
    kghm,
    forced_company="KGHM Polska Miedź"
)


print(
    "KGHM:",
    len(kghm)
)


# 7. WCZYTANIE POZOSTAŁYCH 64 LLM-2

llm2_8 = pd.read_excel(
    LLM2_8_FILE,
    sheet_name="Wyniki"
)


llm2_8 = prepare_llm2(
    llm2_8
)


print(
    "Pozostałe spółki:",
    len(llm2_8)
)


# 8. POŁĄCZENIE LLM-2

llm2 = pd.concat(
    [
        orlen,
        kghm,
        llm2_8
    ],
    ignore_index=True
)


llm2 = llm2.sort_values(
    [
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy"
    ]
).reset_index(
    drop=True
)


print()
print(
    "Łączna liczba LLM-2:",
    len(llm2)
)


if len(llm2) != 80:

    raise ValueError(
        f"LLM-2 powinno mieć 80 rekordów, "
        f"a ma {len(llm2)}."
    )


# 9. KONTROLA DUPLIKATÓW LLM-2

duplicate_llm2 = llm2[
    llm2.duplicated(
        subset=[
            "Spółka rzeczywista",
            "Rok prognozy",
            "Kwartał prognozy"
        ],
        keep=False
    )
]


print(
    "Duplikaty LLM-2:",
    len(duplicate_llm2)
)


if len(duplicate_llm2) > 0:

    display(
        duplicate_llm2
    )

    raise ValueError(
        "LLM-2 zawiera duplikaty."
    )


# 10. KONTROLA BRAKÓW LLM-2

llm2_required_values = [

    "Przychody rzeczywiste",

    "EBIT rzeczywisty",

    "Zysk netto rzeczywisty",

    "LLM2 Przychody",

    "LLM2 EBIT",

    "LLM2 Zysk netto"

]


print()
print(
    "Braki LLM-2:"
)

print(
    llm2[
        llm2_required_values
    ]
    .isna()
    .sum()
)


if (
    llm2[
        llm2_required_values
    ]
    .isna()
    .sum()
    .sum()
    != 0
):

    raise ValueError(
        "W LLM-2 występują brakujące wartości."
    )


# 11. WCZYTANIE LLM-1

llm1 = pd.read_excel(
    LLM1_FILE,
    sheet_name="Wyniki LLM"
)


llm1_required = [

    "Spółka rzeczywista",

    "Rok prognozy",

    "Kwartał prognozy",

    "Przychody rzeczywiste",

    "EBIT rzeczywisty",

    "Zysk netto rzeczywisty",

    "LLM Przychody",

    "LLM EBIT",

    "LLM Zysk netto"

]


missing_llm1 = [

    col

    for col in llm1_required

    if col not in llm1.columns

]


if missing_llm1:

    raise ValueError(
        f"Brak kolumn LLM-1: {missing_llm1}"
    )


llm1 = llm1[
    llm1_required
].copy()


llm1 = llm1.rename(
    columns={

        "LLM Przychody":
            "LLM1 Przychody",

        "LLM EBIT":
            "LLM1 EBIT",

        "LLM Zysk netto":
            "LLM1 Zysk netto"

    }
)


for col in [

    "Rok prognozy",

    "Przychody rzeczywiste",

    "EBIT rzeczywisty",

    "Zysk netto rzeczywisty",

    "LLM1 Przychody",

    "LLM1 EBIT",

    "LLM1 Zysk netto"

]:

    llm1[col] = pd.to_numeric(
        llm1[col],
        errors="coerce"
    )


llm1[
    "Kwartał prognozy"
] = (
    llm1[
        "Kwartał prognozy"
    ]
    .astype(str)
    .str.strip()
)


print()
print(
    "Liczba LLM-1:",
    len(llm1)
)


if len(llm1) != 80:

    raise ValueError(
        "LLM-1 powinno mieć 80 rekordów."
    )


# 12. KONTROLA, CZY LLM-1 I LLM-2 MAJĄ TE SAME OBSERWACJE

keys = [

    "Spółka rzeczywista",

    "Rok prognozy",

    "Kwartał prognozy"

]


keys_llm1 = set(
    map(
        tuple,
        llm1[
            keys
        ].values.tolist()
    )
)


keys_llm2 = set(
    map(
        tuple,
        llm2[
            keys
        ].values.tolist()
    )
)


only_llm1 = (
    keys_llm1
    - keys_llm2
)


only_llm2 = (
    keys_llm2
    - keys_llm1
)


print(
    "Obserwacje tylko w LLM-1:",
    len(only_llm1)
)


print(
    "Obserwacje tylko w LLM-2:",
    len(only_llm2)
)


if (
    len(only_llm1) > 0
    or
    len(only_llm2) > 0
):

    print(
        "Tylko LLM1:",
        only_llm1
    )

    print(
        "Tylko LLM2:",
        only_llm2
    )

    raise ValueError(
        "LLM-1 i LLM-2 nie dotyczą identycznych obserwacji."
    )


# 13. DEFINICJA ZMIENNYCH

VARIABLES_LLM1 = {

    "Przychody": (
        "Przychody rzeczywiste",
        "LLM1 Przychody"
    ),

    "EBIT": (
        "EBIT rzeczywisty",
        "LLM1 EBIT"
    ),

    "Zysk netto": (
        "Zysk netto rzeczywisty",
        "LLM1 Zysk netto"
    )

}


VARIABLES_LLM2 = {

    "Przychody": (
        "Przychody rzeczywiste",
        "LLM2 Przychody"
    ),

    "EBIT": (
        "EBIT rzeczywisty",
        "LLM2 EBIT"
    ),

    "Zysk netto": (
        "Zysk netto rzeczywisty",
        "LLM2 Zysk netto"
    )

}


# 14. FUNKCJA LICZENIA METRYK DLA MODELU

def calculate_model_metrics(
    df,
    variables,
    model_name
):

    rows = []


    for variable, (
        actual_col,
        predicted_col
    ) in variables.items():


        temp = df[
            [
                actual_col,
                predicted_col
            ]
        ].dropna()


        y_true = (
            temp[
                actual_col
            ]
            .to_numpy(
                dtype=float
            )
        )


        y_pred = (
            temp[
                predicted_col
            ]
            .to_numpy(
                dtype=float
            )
        )


        rows.append({

            "Zmienna":
                variable,

            "Model":
                model_name,

            "Liczba obserwacji":
                len(temp),

            "MAE":
                mae(
                    y_true,
                    y_pred
                ),

            "RMSE":
                rmse(
                    y_true,
                    y_pred
                ),

            "sMAPE (%)":
                smape(
                    y_true,
                    y_pred
                ),

            "WAPE (%)":
                wape(
                    y_true,
                    y_pred
                )

        })


    return pd.DataFrame(
        rows
    )


# 15. METRYKI LLM-1

metrics_llm1 = calculate_model_metrics(

    llm1,

    VARIABLES_LLM1,

    "LLM - liczby"

)

# 16. METRYKI LLM-2

metrics_llm2 = calculate_model_metrics(

    llm2,

    VARIABLES_LLM2,

    "LLM - liczby + tekst"

)


print()
print("=" * 70)
print("FINALNE WYNIKI LLM-2")
print("=" * 70)

display(
    metrics_llm2
)


# 17. WCZYTANIE WYNIKÓW ML

xls_ml = pd.ExcelFile(
    ML_FILE
)


ml_results = None

ml_sheet = None


for sheet in xls_ml.sheet_names:

    temp = pd.read_excel(
        ML_FILE,
        sheet_name=sheet
    )


    required = {
        "Zmienna",
        "Model",
        "MAE",
        "RMSE"
    }


    if required.issubset(
        set(temp.columns)
    ):

        ml_results = temp.copy()

        ml_sheet = sheet

        break


if ml_results is None:

    raise ValueError(
        "Nie znaleziono arkusza z wynikami ML."
    )


print()
print(
    "Wyniki ML odczytano z arkusza:",
    ml_sheet
)


# 18. NORMALIZACJA ML

ml_results[
    "Zmienna"
] = (
    ml_results[
        "Zmienna"
    ]
    .apply(
        normalize_variable
    )
)


# interesują nas trzy modele
ml_results = ml_results[

    ml_results[
        "Model"
    ].isin(
        [
            "Seasonal Naive",
            "Linear Regression",
            "XGBoost"
        ]
    )

].copy()


# kolumna liczby obserwacji może nie istnieć
if (
    "Liczba obserwacji"
    not in ml_results.columns
):

    ml_results[
        "Liczba obserwacji"
    ] = 80


# upewniamy się, że metryki istnieją
for metric in [
    "MAE",
    "RMSE",
    "sMAPE (%)",
    "WAPE (%)"
]:

    if metric not in ml_results.columns:

        ml_results[
            metric
        ] = np.nan


ml_results = ml_results[
    [
        "Zmienna",
        "Model",
        "Liczba obserwacji",
        "MAE",
        "RMSE",
        "sMAPE (%)",
        "WAPE (%)"
    ]
]

# 19. FINALNA TABELA 5 MODELI

all_metrics = pd.concat(

    [
        ml_results,
        metrics_llm1,
        metrics_llm2
    ],

    ignore_index=True

)


model_order = {

    "Seasonal Naive": 1,

    "Linear Regression": 2,

    "XGBoost": 3,

    "LLM - liczby": 4,

    "LLM - liczby + tekst": 5

}


variable_order = {

    "Przychody": 1,

    "EBIT": 2,

    "Zysk netto": 3

}


all_metrics[
    "_model_order"
] = (
    all_metrics[
        "Model"
    ]
    .map(
        model_order
    )
)


all_metrics[
    "_variable_order"
] = (
    all_metrics[
        "Zmienna"
    ]
    .map(
        variable_order
    )
)


all_metrics = all_metrics.sort_values(

    [
        "_variable_order",
        "_model_order"
    ]

).drop(
    columns=[
        "_model_order",
        "_variable_order"
    ]
).reset_index(
    drop=True
)


print()
print("=" * 70)
print("WSZYSTKIE MODELE")
print("=" * 70)

display(
    all_metrics
)


# 20. PORÓWNANIE LLM-1 VS LLM-2

llm_comparison = pd.merge(

    metrics_llm1,

    metrics_llm2,

    on="Zmienna",

    suffixes=(
        " LLM1",
        " LLM2"
    )

)


comparison_rows = []


for _, row in llm_comparison.iterrows():

    result = {

        "Zmienna":
            row[
                "Zmienna"
            ]
    }


    for metric in [

        "MAE",
        "RMSE",
        "sMAPE (%)",
        "WAPE (%)"

    ]:


        value1 = row[
            metric + " LLM1"
        ]


        value2 = row[
            metric + " LLM2"
        ]


        result[
            metric + " LLM-1"
        ] = value1


        result[
            metric + " LLM-2"
        ] = value2


        if (
            pd.notna(value1)
            and
            value1 != 0
        ):

            improvement = (

                (
                    value1
                    - value2
                )

                /

                abs(
                    value1
                )

                * 100
            )

        else:

            improvement = np.nan


        result[
            "Poprawa "
            + metric
            + " (%)"
        ] = improvement


    comparison_rows.append(
        result
    )


llm_improvement = pd.DataFrame(
    comparison_rows
)


print()
print("=" * 70)
print("LLM-1 VS LLM-2")
print("=" * 70)

display(
    llm_improvement
)


# 21. NAJLEPSZY MODEL DLA KAŻDEJ METRYKI

winner_rows = []


for variable in [

    "Przychody",
    "EBIT",
    "Zysk netto"

]:

    temp = all_metrics[
        all_metrics[
            "Zmienna"
        ] == variable
    ]


    for metric in [

        "MAE",
        "RMSE",
        "sMAPE (%)",
        "WAPE (%)"

    ]:


        valid = temp.dropna(
            subset=[
                metric
            ]
        )


        if len(valid) == 0:
            continue


        min_value = valid[
            metric
        ].min()


        winners = valid[
            np.isclose(
                valid[
                    metric
                ],
                min_value
            )
        ][
            "Model"
        ].tolist()


        winner_rows.append({

            "Zmienna":
                variable,

            "Metryka":
                metric,

            "Najlepszy model":
                " / ".join(
                    winners
                ),

            "Wartość":
                min_value

        })


winners = pd.DataFrame(
    winner_rows
)


print()
print("=" * 70)
print("NAJLEPSZE MODELE")
print("=" * 70)

display(
    winners
)


# 22. LICZBA ZWYCIĘSTW MODELI

win_counts = (

    winners[
        "Najlepszy model"
    ]
    .value_counts()
    .reset_index()

)


win_counts.columns = [

    "Model",

    "Liczba zwycięstw"

]


print()
print(
    "Liczba zwycięstw:"
)

display(
    win_counts
)


# 23. METRYKI LLM-2 DLA KAŻDEJ SPÓŁKI

company_rows = []


for company in sorted(
    llm2[
        "Spółka rzeczywista"
    ].unique()
):

    company_df = llm2[
        llm2[
            "Spółka rzeczywista"
        ] == company
    ]


    for variable, (
        actual_col,
        pred_col
    ) in VARIABLES_LLM2.items():


        y_true = company_df[
            actual_col
        ].to_numpy(
            dtype=float
        )


        y_pred = company_df[
            pred_col
        ].to_numpy(
            dtype=float
        )


        company_rows.append({

            "Spółka":
                company,

            "Zmienna":
                variable,

            "N":
                len(
                    company_df
                ),

            "MAE":
                mae(
                    y_true,
                    y_pred
                ),

            "RMSE":
                rmse(
                    y_true,
                    y_pred
                ),

            "sMAPE (%)":
                smape(
                    y_true,
                    y_pred
                ),

            "WAPE (%)":
                wape(
                    y_true,
                    y_pred
                )

        })


llm2_by_company = pd.DataFrame(
    company_rows
)

# 24. PORÓWNANIE LLM-1 VS LLM-2 DLA KAŻDEJ SPÓŁKI

paired = pd.merge(

    llm1,

    llm2,

    on=[
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy"
    ],

    suffixes=(
        " LLM1",
        " LLM2"
    )

)


company_comparison_rows = []


for company in sorted(
    paired[
        "Spółka rzeczywista"
    ].unique()
):

    cdf = paired[
        paired[
            "Spółka rzeczywista"
        ] == company
    ]


    configurations = {

        "Przychody": (

            "Przychody rzeczywiste LLM1",

            "LLM1 Przychody",

            "LLM2 Przychody"

        ),

        "EBIT": (

            "EBIT rzeczywisty LLM1",

            "LLM1 EBIT",

            "LLM2 EBIT"

        ),

        "Zysk netto": (

            "Zysk netto rzeczywisty LLM1",

            "LLM1 Zysk netto",

            "LLM2 Zysk netto"

        )

    }


    for variable, (
        actual_col,
        llm1_col,
        llm2_col
    ) in configurations.items():


        y_true = cdf[
            actual_col
        ].to_numpy(
            dtype=float
        )


        pred1 = cdf[
            llm1_col
        ].to_numpy(
            dtype=float
        )


        pred2 = cdf[
            llm2_col
        ].to_numpy(
            dtype=float
        )


        rmse1 = rmse(
            y_true,
            pred1
        )


        rmse2 = rmse(
            y_true,
            pred2
        )


        improvement = (

            (
                rmse1
                - rmse2
            )

            /

            rmse1

            * 100

            if rmse1 != 0

            else np.nan
        )


        company_comparison_rows.append({

            "Spółka":
                company,

            "Zmienna":
                variable,

            "RMSE LLM-1":
                rmse1,

            "RMSE LLM-2":
                rmse2,

            "Poprawa RMSE (%)":
                improvement

        })


llm_company_comparison = pd.DataFrame(
    company_comparison_rows
)


# 25. KONTROLA FINALNA

control = pd.DataFrame({

    "Element": [

        "Liczba obserwacji LLM-1",

        "Liczba obserwacji LLM-2",

        "Liczba spółek",

        "Okres testowy",

        "Liczba modeli",

        "Liczba zmiennych",

        "Braki LLM-1",

        "Braki LLM-2"

    ],


    "Wartość": [

        len(
            llm1
        ),

        len(
            llm2
        ),

        llm2[
            "Spółka rzeczywista"
        ].nunique(),

        "2023-2024",

        5,

        3,

        int(
            llm1.isna()
            .sum()
            .sum()
        ),

        int(
            llm2.isna()
            .sum()
            .sum()
        )

    ]

})


# 26. ZAPIS FINALNEGO EXCELA

with pd.ExcelWriter(

    OUTPUT_FILE,

    engine="openpyxl"

) as writer:


    # 80 obserwacji LLM-2
    llm2.to_excel(

        writer,

        sheet_name="LLM2 - 80 prognoz",

        index=False

    )


    # wyniki LLM2
    metrics_llm2.to_excel(

        writer,

        sheet_name="Metryki LLM2",

        index=False

    )


    # wszystkie modele
    all_metrics.to_excel(

        writer,

        sheet_name="Wszystkie modele",

        index=False

    )


    # LLM1 vs LLM2
    llm_improvement.to_excel(

        writer,

        sheet_name="LLM1 vs LLM2",

        index=False

    )


    # zwycięzcy
    winners.to_excel(

        writer,

        sheet_name="Najlepsze modele",

        index=False

    )


    # liczba zwycięstw
    win_counts.to_excel(

        writer,

        sheet_name="Liczba zwycięstw",

        index=False

    )


    # wyniki LLM2 dla spółek
    llm2_by_company.to_excel(

        writer,

        sheet_name="LLM2 wg spółek",

        index=False

    )


    # wpływ tekstu wg spółek
    llm_company_comparison.to_excel(

        writer,

        sheet_name="Wpływ tekstu wg spółek",

        index=False

    )


    # kontrola
    control.to_excel(

        writer,

        sheet_name="Kontrola",

        index=False

    )

# 27. PODSUMOWANIE

print()
print("=" * 70)
print("FINALNA ANALIZA ZAKOŃCZONA")
print("=" * 70)

print()

print(
    "LLM-1:",
    len(llm1),
    "prognoz"
)

print(
    "LLM-2:",
    len(llm2),
    "prognoz"
)

print(
    "Spółek:",
    llm2[
        "Spółka rzeczywista"
    ].nunique()
)

print()

print(
    "Plik wynikowy:"
)

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)

print()
print(
    "Najważniejsza tabela:"
)

display(
    all_metrics
)



In [ ]:
# Analiza wyników według spółek i liczby zwycięstw

# Dane:
# FINALNE_wyniki_badania.xlsx
# Wynik:
# ANALIZA_spolki_i_zwyciestwa.xlsx

import os
import numpy as np
import pandas as pd

# 1. PLIKI

FINAL_FILE = "FINALNE_wyniki_badania.xlsx"

LLM1_FILE = "wyniki_LLM_liczby_v3.xlsx"

OUTPUT_FILE = "ANALIZA_spolki_i_zwyciestwa.xlsx"

# 2. FUNKCJE METRYK

def mae(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.mean(
        np.abs(
            y_true - y_pred
        )
    )


def rmse(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )


def smape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denominator = (
        np.abs(y_true)
        +
        np.abs(y_pred)
    )

    mask = (
        denominator != 0
    )

    if mask.sum() == 0:
        return np.nan

    return (
        np.mean(
            2
            * np.abs(
                y_pred[mask]
                - y_true[mask]
            )
            / denominator[mask]
        )
        * 100
    )


def wape(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    denominator = np.sum(
        np.abs(y_true)
    )

    if denominator == 0:
        return np.nan

    return (
        np.sum(
            np.abs(
                y_true - y_pred
            )
        )
        /
        denominator
        * 100
    )

# 3. WCZYTANIE 80 PROGNOZ LLM-2

llm2 = pd.read_excel(
    FINAL_FILE,
    sheet_name="LLM2 - 80 prognoz"
)

print(
    "Liczba obserwacji LLM-2:",
    len(llm2)
)


if len(llm2) != 80:

    raise ValueError(
        "LLM-2 powinno zawierać 80 obserwacji."
    )

# 4. WCZYTANIE LLM-1

llm1 = pd.read_excel(
    LLM1_FILE,
    sheet_name="Wyniki LLM"
)

print(
    "Liczba obserwacji LLM-1:",
    len(llm1)
)


if len(llm1) != 80:

    raise ValueError(
        "LLM-1 powinno zawierać 80 obserwacji."
    )

# 5. WYBÓR KOLUMN LLM-1

llm1 = llm1[
    [
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy",

        "Przychody rzeczywiste",
        "EBIT rzeczywisty",
        "Zysk netto rzeczywisty",

        "LLM Przychody",
        "LLM EBIT",
        "LLM Zysk netto"
    ]
].copy()


llm1 = llm1.rename(
    columns={

        "LLM Przychody":
            "LLM1 Przychody",

        "LLM EBIT":
            "LLM1 EBIT",

        "LLM Zysk netto":
            "LLM1 Zysk netto"

    }
)

# 6. ŁĄCZENIE LLM-1 I LLM-2

paired = pd.merge(

    llm1,

    llm2[
        [
            "Spółka rzeczywista",
            "Rok prognozy",
            "Kwartał prognozy",

            "LLM2 Przychody",
            "LLM2 EBIT",
            "LLM2 Zysk netto"
        ]
    ],

    on=[
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy"
    ],

    how="inner"
)


print(
    "Liczba połączonych obserwacji:",
    len(paired)
)


if len(paired) != 80:

    raise ValueError(
        "Połączenie LLM-1 i LLM-2 powinno dać 80 rekordów."
    )

# 7. DEFINICJA ZMIENNYCH

VARIABLES = {

    "Przychody": {

        "actual":
            "Przychody rzeczywiste",

        "LLM1":
            "LLM1 Przychody",

        "LLM2":
            "LLM2 Przychody"

    },

    "EBIT": {

        "actual":
            "EBIT rzeczywisty",

        "LLM1":
            "LLM1 EBIT",

        "LLM2":
            "LLM2 EBIT"

    },

    "Zysk netto": {

        "actual":
            "Zysk netto rzeczywisty",

        "LLM1":
            "LLM1 Zysk netto",

        "LLM2":
            "LLM2 Zysk netto"

    }

}

# 8. METRYKI LLM-1 I LLM-2 DLA KAŻDEJ SPÓŁKI

company_rows = []


companies = sorted(
    paired[
        "Spółka rzeczywista"
    ].unique()
)


for company in companies:

    temp_company = paired[
        paired[
            "Spółka rzeczywista"
        ] == company
    ]


    for variable, cols in VARIABLES.items():

        y_true = temp_company[
            cols["actual"]
        ].to_numpy(
            dtype=float
        )


        for model_code, model_name in [

            (
                "LLM1",
                "LLM - liczby"
            ),

            (
                "LLM2",
                "LLM - liczby + tekst"
            )

        ]:

            y_pred = temp_company[
                cols[
                    model_code
                ]
            ].to_numpy(
                dtype=float
            )


            company_rows.append({

                "Spółka":
                    company,

                "Zmienna":
                    variable,

                "Model":
                    model_name,

                "N":
                    len(
                        temp_company
                    ),

                "MAE":
                    mae(
                        y_true,
                        y_pred
                    ),

                "RMSE":
                    rmse(
                        y_true,
                        y_pred
                    ),

                "sMAPE (%)":
                    smape(
                        y_true,
                        y_pred
                    ),

                "WAPE (%)":
                    wape(
                        y_true,
                        y_pred
                    )

            })


company_metrics = pd.DataFrame(
    company_rows
)


print()
print("=" * 70)
print("LLM-1 I LLM-2 WG SPÓŁEK")
print("=" * 70)

display(
    company_metrics
)

# 9. WPŁYW DODANIA TEKSTU DLA KAŻDEJ SPÓŁKI

improvement_rows = []


for company in companies:

    for variable in VARIABLES.keys():

        temp = company_metrics[
            (
                company_metrics[
                    "Spółka"
                ] == company
            )
            &
            (
                company_metrics[
                    "Zmienna"
                ] == variable
            )
        ]


        llm1_row = temp[
            temp[
                "Model"
            ] == "LLM - liczby"
        ].iloc[0]


        llm2_row = temp[
            temp[
                "Model"
            ] == "LLM - liczby + tekst"
        ].iloc[0]


        result = {

            "Spółka":
                company,

            "Zmienna":
                variable

        }


        for metric in [

            "MAE",
            "RMSE",
            "sMAPE (%)",
            "WAPE (%)"

        ]:

            value1 = llm1_row[
                metric
            ]

            value2 = llm2_row[
                metric
            ]


            if (
                pd.notna(value1)
                and
                value1 != 0
            ):

                improvement = (

                    (
                        value1
                        - value2
                    )

                    /
                    abs(
                        value1
                    )

                    * 100
                )

            else:

                improvement = np.nan


            result[
                metric + " LLM-1"
            ] = value1


            result[
                metric + " LLM-2"
            ] = value2


            result[
                "Poprawa "
                + metric
                + " (%)"
            ] = improvement


        improvement_rows.append(
            result
        )


improvement_company = pd.DataFrame(
    improvement_rows
)


print()
print("=" * 70)
print("WPŁYW TEKSTU WG SPÓŁEK")
print("=" * 70)

display(
    improvement_company
)

# 10. CZY TEKST POPRAWIŁ RMSE?

improvement_company[
    "Tekst poprawił RMSE"
] = np.where(

    improvement_company[
        "Poprawa RMSE (%)"
    ] > 0,

    "TAK",

    "NIE"

)

# 11. LICZBA SPÓŁEK, W KTÓRYCH TEKST POPRAWIŁ RMSE

text_win_summary = (

    improvement_company
    .groupby(
        "Zmienna"
    )[
        "Tekst poprawił RMSE"
    ]
    .value_counts()
    .unstack(
        fill_value=0
    )
    .reset_index()

)


print()
print("=" * 70)
print("W ILU SPÓŁKACH TEKST POPRAWIŁ RMSE?")
print("=" * 70)

display(
    text_win_summary
)

# 12. ŚREDNIA I MEDIANA POPRAWY RMSE

text_improvement_summary = (

    improvement_company
    .groupby(
        "Zmienna"
    )
    .agg(

        Średnia_poprawa_RMSE=(
            "Poprawa RMSE (%)",
            "mean"
        ),

        Mediana_poprawy_RMSE=(
            "Poprawa RMSE (%)",
            "median"
        ),

        Min_poprawa_RMSE=(
            "Poprawa RMSE (%)",
            "min"
        ),

        Max_poprawa_RMSE=(
            "Poprawa RMSE (%)",
            "max"
        )

    )
    .reset_index()

)


print()
print("=" * 70)
print("ROZKŁAD POPRAWY RMSE")
print("=" * 70)

display(
    text_improvement_summary
)

# 13. GLOBALNE WYNIKI 5 MODELI

all_metrics = pd.read_excel(
    FINAL_FILE,
    sheet_name="Wszystkie modele"
)

# 14. LICZBA ZWYCIĘSTW

METRICS = [

    "MAE",
    "RMSE",
    "sMAPE (%)",
    "WAPE (%)"

]


winner_rows = []


for variable in [

    "Przychody",
    "EBIT",
    "Zysk netto"

]:

    temp_variable = all_metrics[
        all_metrics[
            "Zmienna"
        ] == variable
    ]


    for metric in METRICS:

        valid = temp_variable.dropna(
            subset=[
                metric
            ]
        )


        min_value = valid[
            metric
        ].min()


        winners = valid[
            np.isclose(
                valid[
                    metric
                ],
                min_value
            )
        ]


        for _, winner in winners.iterrows():

            winner_rows.append({

                "Zmienna":
                    variable,

                "Metryka":
                    metric,

                "Model":
                    winner[
                        "Model"
                    ],

                "Wartość":
                    winner[
                        metric
                    ]

            })


wins_detail = pd.DataFrame(
    winner_rows
)

# 15. PODSUMOWANIE ZWYCIĘSTW

wins_summary = (

    wins_detail[
        "Model"
    ]
    .value_counts()
    .rename_axis(
        "Model"
    )
    .reset_index(
        name="Liczba zwycięstw"
    )

)


# wszystkie modele, nawet jeśli mają zero
all_models = pd.DataFrame({

    "Model": [

        "Seasonal Naive",

        "Linear Regression",

        "XGBoost",

        "LLM - liczby",

        "LLM - liczby + tekst"

    ]

})


wins_summary = pd.merge(

    all_models,

    wins_summary,

    on="Model",

    how="left"

)


wins_summary[
    "Liczba zwycięstw"
] = (
    wins_summary[
        "Liczba zwycięstw"
    ]
    .fillna(0)
    .astype(int)
)


wins_summary[
    "Udział zwycięstw (%)"
] = (

    wins_summary[
        "Liczba zwycięstw"
    ]

    /

    12

    * 100

)


wins_summary = wins_summary.sort_values(

    "Liczba zwycięstw",

    ascending=False

).reset_index(
    drop=True
)


print()
print("=" * 70)
print("LICZBA ZWYCIĘSTW MODELI")
print("=" * 70)

display(
    wins_summary
)


print()
print(
    "Szczegóły zwycięstw:"
)

display(
    wins_detail
)

# 16. RANKING MODELI DLA KAŻDEJ ZMIENNEJ

rmse_ranking = all_metrics[
    [
        "Zmienna",
        "Model",
        "RMSE"
    ]
].copy()


rmse_ranking[
    "Pozycja wg RMSE"
] = (

    rmse_ranking
    .groupby(
        "Zmienna"
    )[
        "RMSE"
    ]
    .rank(
        method="min",
        ascending=True
    )
    .astype(int)

)


rmse_ranking = rmse_ranking.sort_values(

    [
        "Zmienna",
        "Pozycja wg RMSE"
    ]

)


print()
print("=" * 70)
print("RANKING WG RMSE")
print("=" * 70)

display(
    rmse_ranking
)

# 17. ZAPIS DO EXCELA

with pd.ExcelWriter(

    OUTPUT_FILE,

    engine="openpyxl"

) as writer:


    company_metrics.to_excel(

        writer,

        sheet_name="LLM wg spółek",

        index=False

    )


    improvement_company.to_excel(

        writer,

        sheet_name="Wpływ tekstu",

        index=False

    )


    text_win_summary.to_excel(

        writer,

        sheet_name="Tekst - liczba popraw",

        index=False

    )


    text_improvement_summary.to_excel(

        writer,

        sheet_name="Tekst - podsumowanie",

        index=False

    )


    wins_summary.to_excel(

        writer,

        sheet_name="Zwycięstwa modeli",

        index=False

    )


    wins_detail.to_excel(

        writer,

        sheet_name="Szczegóły zwycięstw",

        index=False

    )


    rmse_ranking.to_excel(

        writer,

        sheet_name="Ranking RMSE",

        index=False

    )

# 18. KONIEC

print()
print("=" * 70)
print("ANALIZA ZAKOŃCZONA")
print("=" * 70)

print()

print(
    "Plik:"
)

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)



In [ ]:
# Analiza odporności: Wilcoxon, Holm i bootstrap klastrowy

import os
import numpy as np
import pandas as pd

from scipy.stats import wilcoxon

# 1. PLIKI

FINAL_FILE = "FINALNE_wyniki_badania.xlsx"

LLM1_FILE = "wyniki_LLM_liczby_v3.xlsx"

OUTPUT_FILE = "ANALIZA_odpornosci_LLM.xlsx"

# 2. USTAWIENIA BOOTSTRAPU

N_BOOTSTRAP = 10000

RANDOM_SEED = 140343

rng = np.random.default_rng(
    RANDOM_SEED
)

# 3. FUNKCJE METRYK

def mae(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.mean(
        np.abs(
            y_true - y_pred
        )
    )


def rmse(y_true, y_pred):

    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    return np.sqrt(
        np.mean(
            (y_true - y_pred) ** 2
        )
    )

# 4. KOREKTA HOLMA

def holm_adjust(p_values):

    p_values = np.asarray(
        p_values,
        dtype=float
    )

    n = len(
        p_values
    )

    order = np.argsort(
        p_values
    )

    sorted_p = p_values[
        order
    ]

    adjusted_sorted = np.zeros(
        n
    )

    previous = 0.0

    for i, p in enumerate(
        sorted_p
    ):

        value = (
            (n - i)
            * p
        )

        value = max(
            value,
            previous
        )

        value = min(
            value,
            1.0
        )

        adjusted_sorted[
            i
        ] = value

        previous = value


    adjusted = np.empty(
        n
    )

    adjusted[
        order
    ] = adjusted_sorted

    return adjusted

# 5. BEZPIECZNY TEST WILCOXONA

def safe_wilcoxon(
    x,
    y
):

    x = np.asarray(
        x,
        dtype=float
    )

    y = np.asarray(
        y,
        dtype=float
    )

    mask = (
        np.isfinite(x)
        &
        np.isfinite(y)
    )

    x = x[
        mask
    ]

    y = y[
        mask
    ]

    differences = (
        x - y
    )


    # jeśli wszystkie różnice = 0
    if np.allclose(
        differences,
        0
    ):

        return (
            np.nan,
            1.0
        )


    result = wilcoxon(
        x,
        y,
        alternative="two-sided",
        zero_method="wilcox"
    )


    return (
        result.statistic,
        result.pvalue
    )

# 6. WCZYTANIE LLM-2

llm2 = pd.read_excel(
    FINAL_FILE,
    sheet_name="LLM2 - 80 prognoz"
)


print(
    "LLM-2:",
    len(llm2)
)

# 7. WCZYTANIE LLM-1

llm1 = pd.read_excel(
    LLM1_FILE,
    sheet_name="Wyniki LLM"
)


print(
    "LLM-1:",
    len(llm1)
)

# 8. PRZYGOTOWANIE LLM-1

llm1 = llm1[
    [
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy",

        "Przychody rzeczywiste",
        "EBIT rzeczywisty",
        "Zysk netto rzeczywisty",

        "LLM Przychody",
        "LLM EBIT",
        "LLM Zysk netto"
    ]
].copy()


llm1 = llm1.rename(
    columns={

        "LLM Przychody":
            "LLM1 Przychody",

        "LLM EBIT":
            "LLM1 EBIT",

        "LLM Zysk netto":
            "LLM1 Zysk netto"

    }
)

# 9. POŁĄCZENIE 80 OBSERWACJI

paired = pd.merge(

    llm1,

    llm2[
        [
            "Spółka rzeczywista",
            "Rok prognozy",
            "Kwartał prognozy",

            "LLM2 Przychody",
            "LLM2 EBIT",
            "LLM2 Zysk netto"
        ]
    ],

    on=[
        "Spółka rzeczywista",
        "Rok prognozy",
        "Kwartał prognozy"
    ],

    how="inner"
)


print(
    "Połączone obserwacje:",
    len(paired)
)


if len(paired) != 80:

    raise ValueError(
        "Powinno być dokładnie 80 obserwacji."
    )


print(
    "Liczba spółek:",
    paired[
        "Spółka rzeczywista"
    ].nunique()
)

# 10. KONFIGURACJA ZMIENNYCH

VARIABLES = {

    "Przychody": {

        "actual":
            "Przychody rzeczywiste",

        "LLM1":
            "LLM1 Przychody",

        "LLM2":
            "LLM2 Przychody"

    },


    "EBIT": {

        "actual":
            "EBIT rzeczywisty",

        "LLM1":
            "LLM1 EBIT",

        "LLM2":
            "LLM2 EBIT"

    },


    "Zysk netto": {

        "actual":
            "Zysk netto rzeczywisty",

        "LLM1":
            "LLM1 Zysk netto",

        "LLM2":
            "LLM2 Zysk netto"

    }

}

# 11. BŁĘDY DLA KAŻDEJ OBSERWACJI

for variable, cols in VARIABLES.items():

    safe_name = (
        variable
        .replace(
            " ",
            "_"
        )
    )


    paired[
        f"AE_LLM1_{safe_name}"
    ] = np.abs(

        paired[
            cols["actual"]
        ]
        -
        paired[
            cols["LLM1"]
        ]

    )


    paired[
        f"AE_LLM2_{safe_name}"
    ] = np.abs(

        paired[
            cols["actual"]
        ]
        -
        paired[
            cols["LLM2"]
        ]

    )


    paired[
        f"SE_LLM1_{safe_name}"
    ] = (

        paired[
            cols["actual"]
        ]
        -
        paired[
            cols["LLM1"]
        ]

    ) ** 2


    paired[
        f"SE_LLM2_{safe_name}"
    ] = (

        paired[
            cols["actual"]
        ]
        -
        paired[
            cols["LLM2"]
        ]

    ) ** 2

# 12. ANALIZA NA POZIOMIE SPÓŁKI

company_rows = []


for company in sorted(
    paired[
        "Spółka rzeczywista"
    ].unique()
):

    cdf = paired[
        paired[
            "Spółka rzeczywista"
        ] == company
    ]


    for variable, cols in VARIABLES.items():

        y_true = cdf[
            cols["actual"]
        ].to_numpy(
            dtype=float
        )


        pred1 = cdf[
            cols["LLM1"]
        ].to_numpy(
            dtype=float
        )


        pred2 = cdf[
            cols["LLM2"]
        ].to_numpy(
            dtype=float
        )


        mae1 = mae(
            y_true,
            pred1
        )


        mae2 = mae(
            y_true,
            pred2
        )


        rmse1 = rmse(
            y_true,
            pred1
        )


        rmse2 = rmse(
            y_true,
            pred2
        )


        company_rows.append({

            "Spółka":
                company,

            "Zmienna":
                variable,

            "N":
                len(cdf),

            "MAE LLM-1":
                mae1,

            "MAE LLM-2":
                mae2,

            "Różnica MAE":
                mae1 - mae2,

            "RMSE LLM-1":
                rmse1,

            "RMSE LLM-2":
                rmse2,

            "Różnica RMSE":
                rmse1 - rmse2,

            "Poprawa RMSE (%)":

                (
                    (rmse1 - rmse2)
                    /
                    rmse1
                    * 100
                )

                if rmse1 != 0

                else np.nan

        })


company_metrics = pd.DataFrame(
    company_rows
)

# 13. GŁÓWNY TEST WILCOXONA

wilcoxon_company_rows = []


for variable in VARIABLES.keys():

    temp = company_metrics[
        company_metrics[
            "Zmienna"
        ] == variable
    ]


    # MAE
    
    stat_mae, p_mae = safe_wilcoxon(

        temp[
            "MAE LLM-1"
        ],

        temp[
            "MAE LLM-2"
        ]

    )


    # RMSE
    
    stat_rmse, p_rmse = safe_wilcoxon(

        temp[
            "RMSE LLM-1"
        ],

        temp[
            "RMSE LLM-2"
        ]

    )


    wilcoxon_company_rows.append({

        "Zmienna":
            variable,

        "N spółek":
            len(temp),

        "Statystyka Wilcoxona MAE":
            stat_mae,

        "p MAE":
            p_mae,

        "Statystyka Wilcoxona RMSE":
            stat_rmse,

        "p RMSE":
            p_rmse,

        "Spółki z niższym RMSE LLM-2":
            int(
                (
                    temp[
                        "RMSE LLM-2"
                    ]
                    <
                    temp[
                        "RMSE LLM-1"
                    ]
                )
                .sum()
            ),

        "Spółki razem":
            len(temp),

        "Mediana poprawy RMSE (%)":

            temp[
                "Poprawa RMSE (%)"
            ].median()

    })


wilcoxon_company = pd.DataFrame(
    wilcoxon_company_rows
)


# 14. KOREKTA HOLMA

wilcoxon_company[
    "p MAE Holm"
] = holm_adjust(

    wilcoxon_company[
        "p MAE"
    ].values

)


wilcoxon_company[
    "p RMSE Holm"
] = holm_adjust(

    wilcoxon_company[
        "p RMSE"
    ].values

)


wilcoxon_company[
    "Istotne MAE po Holmie (0,05)"
] = np.where(

    wilcoxon_company[
        "p MAE Holm"
    ] < 0.05,

    "TAK",

    "NIE"

)


wilcoxon_company[
    "Istotne RMSE po Holmie (0,05)"
] = np.where(

    wilcoxon_company[
        "p RMSE Holm"
    ] < 0.05,

    "TAK",

    "NIE"

)


print()
print("=" * 70)
print("GŁÓWNY TEST - WILCOXON NA 10 SPÓŁKACH")
print("=" * 70)

display(
    wilcoxon_company
)

# 15. POMOCNICZY WILCOXON NA 80 OBSERWACJACH

wilcoxon_80_rows = []


for variable in VARIABLES.keys():

    safe_name = variable.replace(
        " ",
        "_"
    )


    error1 = paired[
        f"AE_LLM1_{safe_name}"
    ]


    error2 = paired[
        f"AE_LLM2_{safe_name}"
    ]


    stat, p = safe_wilcoxon(
        error1,
        error2
    )


    wins = (
        error2
        <
        error1
    )


    ties = (
        error2
        ==
        error1
    )


    losses = (
        error2
        >
        error1
    )


    wilcoxon_80_rows.append({

        "Zmienna":
            variable,

        "N":
            len(error1),

        "Statystyka":
            stat,

        "p":
            p,

        "LLM-2 lepszy - liczba prognoz":
            int(
                wins.sum()
            ),

        "Remisy":
            int(
                ties.sum()
            ),

        "LLM-2 gorszy - liczba prognoz":
            int(
                losses.sum()
            ),

        "LLM-2 lepszy (%)":

            wins.mean()
            * 100

    })


wilcoxon_80 = pd.DataFrame(
    wilcoxon_80_rows
)


wilcoxon_80[
    "p Holm"
] = holm_adjust(

    wilcoxon_80[
        "p"
    ].values

)


print()
print("=" * 70)
print("TEST POMOCNICZY - 80 PROGNOZ")
print("=" * 70)

display(
    wilcoxon_80
)

# 16. CLUSTER BOOTSTRAP

companies = np.array(
    sorted(
        paired[
            "Spółka rzeczywista"
        ].unique()
    )
)


bootstrap_rows = []


for variable, cols in VARIABLES.items():

    y_true_full = paired[
        cols["actual"]
    ].to_numpy(
        dtype=float
    )


    pred1_full = paired[
        cols["LLM1"]
    ].to_numpy(
        dtype=float
    )


    pred2_full = paired[
        cols["LLM2"]
    ].to_numpy(
        dtype=float
    )


    # Wynik obserwowany
    
    observed_mae1 = mae(
        y_true_full,
        pred1_full
    )


    observed_mae2 = mae(
        y_true_full,
        pred2_full
    )


    observed_rmse1 = rmse(
        y_true_full,
        pred1_full
    )


    observed_rmse2 = rmse(
        y_true_full,
        pred2_full
    )


    observed_delta_mae = (
        observed_mae1
        -
        observed_mae2
    )


    observed_delta_rmse = (
        observed_rmse1
        -
        observed_rmse2
    )


    bootstrap_delta_mae = []

    bootstrap_delta_rmse = []


    # Bootstrap
    
    for b in range(
        N_BOOTSTRAP
    ):

        sampled_companies = rng.choice(

            companies,

            size=len(
                companies
            ),

            replace=True

        )


        boot_parts = []


        for sampled_company in sampled_companies:

            part = paired[
                paired[
                    "Spółka rzeczywista"
                ] == sampled_company
            ].copy()


            boot_parts.append(
                part
            )


        boot_df = pd.concat(
            boot_parts,
            ignore_index=True
        )


        y_true = boot_df[
            cols["actual"]
        ].to_numpy(
            dtype=float
        )


        pred1 = boot_df[
            cols["LLM1"]
        ].to_numpy(
            dtype=float
        )


        pred2 = boot_df[
            cols["LLM2"]
        ].to_numpy(
            dtype=float
        )


        delta_mae = (

            mae(
                y_true,
                pred1
            )

            -

            mae(
                y_true,
                pred2
            )

        )


        delta_rmse = (

            rmse(
                y_true,
                pred1
            )

            -

            rmse(
                y_true,
                pred2
            )

        )


        bootstrap_delta_mae.append(
            delta_mae
        )


        bootstrap_delta_rmse.append(
            delta_rmse
        )


    bootstrap_delta_mae = np.asarray(
        bootstrap_delta_mae
    )


    bootstrap_delta_rmse = np.asarray(
        bootstrap_delta_rmse
    )


    # 95% CI
    
    mae_ci_low = np.percentile(
        bootstrap_delta_mae,
        2.5
    )


    mae_ci_high = np.percentile(
        bootstrap_delta_mae,
        97.5
    )


    rmse_ci_low = np.percentile(
        bootstrap_delta_rmse,
        2.5
    )


    rmse_ci_high = np.percentile(
        bootstrap_delta_rmse,
        97.5
    )


    # Jak często bootstrap wskazuje przewagę LLM-2?
    
    prob_mae = (

        np.mean(
            bootstrap_delta_mae
            > 0
        )

        * 100

    )


    prob_rmse = (

        np.mean(
            bootstrap_delta_rmse
            > 0
        )

        * 100

    )


    bootstrap_rows.append({

        "Zmienna":
            variable,

        "MAE LLM-1":
            observed_mae1,

        "MAE LLM-2":
            observed_mae2,

        "Różnica MAE (LLM1 - LLM2)":
            observed_delta_mae,

        "95% CI MAE - dolna":
            mae_ci_low,

        "95% CI MAE - górna":
            mae_ci_high,

        "Bootstrap: P(LLM-2 lepszy MAE) (%)":
            prob_mae,

        "RMSE LLM-1":
            observed_rmse1,

        "RMSE LLM-2":
            observed_rmse2,

        "Różnica RMSE (LLM1 - LLM2)":
            observed_delta_rmse,

        "95% CI RMSE - dolna":
            rmse_ci_low,

        "95% CI RMSE - górna":
            rmse_ci_high,

        "Bootstrap: P(LLM-2 lepszy RMSE) (%)":
            prob_rmse,

        "CI RMSE w całości > 0":

            "TAK"

            if rmse_ci_low > 0

            else "NIE",

        "CI MAE w całości > 0":

            "TAK"

            if mae_ci_low > 0

            else "NIE"

    })


bootstrap_results = pd.DataFrame(
    bootstrap_rows
)


print()
print("=" * 70)
print("CLUSTER BOOTSTRAP - 95% CI")
print("=" * 70)

display(
    bootstrap_results
)


# 17. PROSTA TABELA DO PRACY

summary_rows = []


for variable in VARIABLES.keys():

    w = wilcoxon_company[
        wilcoxon_company[
            "Zmienna"
        ] == variable
    ].iloc[0]


    b = bootstrap_results[
        bootstrap_results[
            "Zmienna"
        ] == variable
    ].iloc[0]


    r80 = wilcoxon_80[
        wilcoxon_80[
            "Zmienna"
        ] == variable
    ].iloc[0]


    summary_rows.append({

        "Zmienna":
            variable,

        "Spółki z poprawą RMSE":
            (
                f"{int(w['Spółki z niższym RMSE LLM-2'])}"
                f"/"
                f"{int(w['Spółki razem'])}"
            ),

        "Mediana poprawy RMSE (%)":
            w[
                "Mediana poprawy RMSE (%)"
            ],

        "Wilcoxon RMSE p":
            w[
                "p RMSE"
            ],

        "Wilcoxon RMSE p Holm":
            w[
                "p RMSE Holm"
            ],

        "Istotność po Holmie":
            w[
                "Istotne RMSE po Holmie (0,05)"
            ],

        "Bootstrap różnica RMSE":
            b[
                "Różnica RMSE (LLM1 - LLM2)"
            ],

        "Bootstrap CI 95% dolna":
            b[
                "95% CI RMSE - dolna"
            ],

        "Bootstrap CI 95% górna":
            b[
                "95% CI RMSE - górna"
            ],

        "Bootstrap przewaga LLM-2 (%)":
            b[
                "Bootstrap: P(LLM-2 lepszy RMSE) (%)"
            ],

        "LLM-2 lepszy w prognozach (%)":
            r80[
                "LLM-2 lepszy (%)"
            ]

    })


summary = pd.DataFrame(
    summary_rows
)


print()
print("=" * 70)
print("PODSUMOWANIE DO PRACY")
print("=" * 70)

display(
    summary
)


# 18. ZAPIS

with pd.ExcelWriter(

    OUTPUT_FILE,

    engine="openpyxl"

) as writer:


    paired.to_excel(

        writer,

        sheet_name="80 błędów",

        index=False

    )


    company_metrics.to_excel(

        writer,

        sheet_name="Metryki spółek",

        index=False

    )


    wilcoxon_company.to_excel(

        writer,

        sheet_name="Wilcoxon - spółki",

        index=False

    )


    wilcoxon_80.to_excel(

        writer,

        sheet_name="Wilcoxon - 80",

        index=False

    )


    bootstrap_results.to_excel(

        writer,

        sheet_name="Cluster bootstrap",

        index=False

    )


    summary.to_excel(

        writer,

        sheet_name="Podsumowanie",

        index=False

    )


# 19. KONIEC

print()
print("=" * 70)
print("ANALIZA ODPORNOŚCI ZAKOŃCZONA")
print("=" * 70)

print()

print(
    os.path.abspath(
        OUTPUT_FILE
    )
)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# WYKRES 1

models = [
    "Seasonal Naive",
    "Regresja liniowa",
    "XGBoost",
    "LLM-1",
    "LLM-2"
]

smape_przychody = [
    14.5672,
    26.4032,
    10.5362,
    17.3039,
    8.2018
]

smape_ebit = [
    47.9824,
    76.1293,
    53.0552,
    56.8158,
    36.9664
]

smape_zysk = [
    63.9129,
    91.6883,
    63.5592,
    73.1098,
    48.8176
]


x = np.arange(len(models))
width = 0.24


fig, ax = plt.subplots(
    figsize=(10, 6)
)

ax.bar(
    x - width,
    smape_przychody,
    width,
    label="Przychody"
)

ax.bar(
    x,
    smape_ebit,
    width,
    label="EBIT"
)

ax.bar(
    x + width,
    smape_zysk,
    width,
    label="Zysk netto"
)


ax.set_title(
    "Porównanie wartości sMAPE dla analizowanych modeli"
)

ax.set_ylabel(
    "sMAPE (%)"
)

ax.set_xlabel(
    "Model"
)

ax.set_xticks(
    x
)

ax.set_xticklabels(
    models
)

ax.legend()

ax.grid(
    axis="y",
    alpha=0.25
)

fig.tight_layout()


# zapis do pliku
plt.savefig(
    "Wykres_1_sMAPE_modele.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Wykres_1_sMAPE_modele.pdf",
    bbox_inches="tight"
)

plt.show()


In [ ]:
# WYKRES 2

companies = [
    "Asseco Poland",
    "Budimex",
    "CD Projekt",
    "Cyfrowy Polsat",
    "Grupa Kęty",
    "KGHM Polska Miedź",
    "LPP",
    "ORLEN",
    "Orange Polska",
    "PGE"
]


# poprawa RMSE w %
rmse_przychody = [
    39.8560,
    47.3267,
    49.8409,
    66.5697,
    40.5903,
    80.1620,
    25.3815,
    44.7552,
    54.3594,
    38.2242
]


rmse_ebit = [
    41.9804,
    32.6451,
    42.8885,
    -1.4630,
    61.7188,
    3.2514,
    43.7945,
    61.2892,
    7.7936,
    12.6926
]


rmse_zysk = [
    52.9593,
    57.6759,
    23.8652,
    35.0384,
    47.1329,
    3.5460,
    62.5422,
    49.0720,
    19.4740,
    9.1311
]


y = np.arange(
    len(companies)
)

height = 0.24


fig, ax = plt.subplots(
    figsize=(11, 7.5)
)


ax.barh(
    y - height,
    rmse_przychody,
    height,
    label="Przychody"
)

ax.barh(
    y,
    rmse_ebit,
    height,
    label="EBIT"
)

ax.barh(
    y + height,
    rmse_zysk,
    height,
    label="Zysk netto"
)


# linia 0%
ax.axvline(
    0,
    linewidth=1
)


ax.set_title(
    "Zmiana RMSE po rozszerzeniu LLM o informacje tekstowe"
)

ax.set_xlabel(
    "Poprawa RMSE LLM-2 względem LLM-1 (%)"
)

ax.set_ylabel(
    "Spółka"
)

ax.set_yticks(
    y
)

ax.set_yticklabels(
    companies
)

ax.invert_yaxis()

ax.legend()

ax.grid(
    axis="x",
    alpha=0.25
)

fig.tight_layout()


# zapis do pliku
plt.savefig(
    "Wykres_2_poprawa_RMSE_spolki.png",
    dpi=300,
    bbox_inches="tight"
)

plt.savefig(
    "Wykres_2_poprawa_RMSE_spolki.pdf",
    bbox_inches="tight"
)

plt.show()
